<a href="https://colab.research.google.com/github/wnstj1126-debug/-/blob/main/M2_ENC_v1_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===== M2_ENC_v1 Cell 1: 설정 + FULL_CONVERT 게이트 + 데이터셋 구조 확인 (읽기 전용) =====
from google.colab import drive
drive.mount('/content/drive')

import os, json, re, time, hashlib, datetime
from pathlib import Path
import numpy as np, pandas as pd

PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/field_iis3dwb")
DATASET_DIR  = PROJECT_ROOT / "bridge_datasets" / "FEMTO_2048_v1_2"
OUT_ROOT     = PROJECT_ROOT / "bridge_outputs"
CONTRACT_VERSION = "M2_ENC_CONTRACT_v1"
FS, NS, REG_MIN = 20480, 2048, 90
EXPECTED = {"LEARNING": (7534, 6), "TEST": (14127, 11), "FULL_TEST": (17355, 11)}
BASELINE_D025 = {"detected": 6, "mean_lead_h": 0.8356, "mean_reg_far_pct": 0.424}
CRITERIA_D028 = {"detected_min": 6, "mean_lead_h_min": 0.752, "mean_reg_far_pct_max": 0.924}
FOLDS = {"Bearing1_3":0,"Bearing2_3":0,"Bearing1_4":1,"Bearing2_4":1,
         "Bearing1_5":2,"Bearing2_5":2,"Bearing3_3":2,
         "Bearing1_6":3,"Bearing2_6":3,"Bearing1_7":4,"Bearing2_7":4}

def stop(code, **kw):
    raise RuntimeError(f"{code} | {json.dumps(kw, ensure_ascii=False, default=str)[:3000]}")

if not DATASET_DIR.is_dir():
    stop("M2ENC_BLOCKED_DATASET_NOT_FOUND", path=str(DATASET_DIR))

# --- 게이트: FULL_CONVERT_PASS 문자열이 들어 있는 요약 JSON 탐색 ---
def _json_has(o, tok):
    if isinstance(o, dict): return any(_json_has(v, tok) for v in o.values())
    if isinstance(o, list): return any(_json_has(v, tok) for v in o)
    return isinstance(o, str) and tok in o

cands = list(DATASET_DIR.glob("*.json"))
for d in OUT_ROOT.iterdir():
    if d.is_dir() and "full_convert" in d.name.lower():
        cands += list(d.glob("*.json"))
GATE_FILES = []
for p in sorted(set(cands)):
    try:
        if _json_has(json.loads(p.read_text(encoding="utf-8-sig")), "FULL_CONVERT_PASS"):
            GATE_FILES.append(p)
    except Exception as e:
        print("[GATE] 읽기 실패:", p.name, type(e).__name__)
if not GATE_FILES:
    stop("M2ENC_BLOCKED_FULL_CONVERT_GATE", searched=[str(p) for p in cands][:30])
print("[GATE] FULL_CONVERT_PASS 확인:", GATE_FILES[0])

# --- 구조 확인 ---
top = sorted(DATASET_DIR.iterdir())
print(f"\n[DATASET] 최상위 항목 {len(top)}개 (최대 40개 표시)")
for p in top[:40]:
    print("  ", "D" if p.is_dir() else "F", p.name)

INDEX_FILES = sorted(DATASET_DIR.rglob("index.csv"))
NPY_FILES   = sorted(DATASET_DIR.rglob("*.npy"))
print(f"\n[INDEX] index.csv {len(INDEX_FILES)}개")
for p in INDEX_FILES[:5]: print("   ", p.relative_to(DATASET_DIR))
print(f"[NPY] .npy {len(NPY_FILES)}개")
for p in NPY_FILES[:5]: print("   ", p.relative_to(DATASET_DIR))

if INDEX_FILES:
    _df = pd.read_csv(INDEX_FILES[0], nrows=5)
    print("\n[INDEX 열 목록]", list(_df.columns))
    print(_df.head(3).T)
print("\nCell 1 완료")


Mounted at /content/drive
[GATE] FULL_CONVERT_PASS 확인: /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/bridge_outputs/20261024_001446_full_convert_v1_2/FULL_CONVERT_FINAL_SUMMARY.json

[DATASET] 최상위 항목 5개 (최대 40개 표시)
   D FULL_TEST
   D LEARNING
   D TEST
   F dataset_contract.json
   F dataset_manifest.csv

[INDEX] index.csv 28개
    FULL_TEST/Bearing1_3/index.csv
    FULL_TEST/Bearing1_4/index.csv
    FULL_TEST/Bearing1_5/index.csv
    FULL_TEST/Bearing1_6/index.csv
    FULL_TEST/Bearing1_7/index.csv
[NPY] .npy 28개
    FULL_TEST/Bearing1_3/signals.npy
    FULL_TEST/Bearing1_4/signals.npy
    FULL_TEST/Bearing1_5/signals.npy
    FULL_TEST/Bearing1_6/signals.npy
    FULL_TEST/Bearing1_7/signals.npy

[INDEX 열 목록] ['row_index', 'split', 'record_key', 'logical_bearing_id', 'sequence_index', 'csv_file_name', 'source_absolute_path', 'source_file_size_bytes', 'source_modified_time_ns', 'source_quick_fingerprint', 'source_delimiter', 'source_sha256_at_read', 'output_array_relative_path', 

In [ ]:
# ===== M2_ENC_v1 Cell 2: 레코드 표 + 분할(D-026) + fold 고정 =====
def pick(df, cands, what, required=True):
    for c in cands:
        if c in df.columns: return c
    if required:
        stop("M2ENC_COLUMN_NOT_FOUND", what=what, candidates=cands, available=list(df.columns))
    return None

def norm_split(v):
    s = str(v).strip().upper().replace("-", "_").replace(" ", "_")
    return {"FULLTEST": "FULL_TEST", "FULL_TEST_SET": "FULL_TEST",
            "TEST_SET": "TEST", "LEARNING_SET": "LEARNING"}.get(s, s)

def norm_bearing(v):
    m = re.search(r"(\d)_(\d)", str(v))
    return f"Bearing{m.group(1)}_{m.group(2)}" if m else str(v).strip()

# index 읽기 (전역 1개 또는 레코드별 여러 개)
parts = []
for p in INDEX_FILES:
    d = pd.read_csv(p)
    d["_index_file"] = str(p)
    parts.append(d)
if not parts: stop("M2ENC_INDEX_NOT_FOUND", dataset=str(DATASET_DIR))
IDX = pd.concat(parts, ignore_index=True)

c_split = pick(IDX, ["split_normalized","split","_split","source_split"], "split")
c_bear  = pick(IDX, ["logical_bearing_id_n","logical_bearing_id","bearing_id","bearing","_bearing"], "bearing")
c_seq   = pick(IDX, ["sequence_index","source_sequence_index","_sequence","seq"], "sequence")
c_npy   = pick(IDX, ["signals_path","npy_path","record_npy_path","output_npy","output_path","signals_file"], "npy", required=False)
c_row   = pick(IDX, ["row_in_record","npy_row","record_row","row_in_npy","local_row","row_in_signals"], "row", required=False)

IDX["_split"]   = IDX[c_split].map(norm_split)
IDX["_bearing"] = IDX[c_bear].map(norm_bearing)
IDX["_seq"]     = pd.to_numeric(IDX[c_seq], errors="coerce")
if IDX["_seq"].isna().any(): stop("M2ENC_INVALID_SEQUENCE", n=int(IDX["_seq"].isna().sum()))
bad = set(IDX["_split"]) - set(EXPECTED)
if bad: stop("M2ENC_UNKNOWN_SPLIT", values=sorted(bad))
IDX["_record_key"] = IDX["_split"] + "/" + IDX["_bearing"]

# 레코드 → npy 파일 연결
def find_npy(split, bearing, idx_file):
    sib = sorted(Path(idx_file).parent.glob("*.npy")) if len(INDEX_FILES) > 1 else []
    if len(sib) == 1: return sib[0]
    hits = []
    for p in NPY_FILES:
        s = str(p.relative_to(DATASET_DIR)).upper()
        if not re.search(rf"BEARING{bearing[-3]}_{bearing[-1]}(?!\d)", s): continue
        is_full = ("FULL_TEST" in s) or ("FULLTEST" in s)
        if split == "FULL_TEST" and is_full: hits.append(p)
        elif split == "TEST" and ("TEST" in s) and not is_full: hits.append(p)
        elif split == "LEARNING" and "LEARNING" in s: hits.append(p)
    if len(hits) != 1:
        stop("M2ENC_NPY_MAPPING_AMBIGUOUS", record=f"{split}/{bearing}", hits=[str(h) for h in hits])
    return hits[0]

rec_rows, WIN = [], {}
for key, g in IDX.groupby("_record_key", sort=True):
    g = g.sort_values("_seq")
    split, bearing = g["_split"].iloc[0], g["_bearing"].iloc[0]
    if c_npy:
        vals = g[c_npy].astype(str).str.strip().unique()
        if len(vals) != 1: stop("M2ENC_NPY_COLUMN_NOT_UNIQUE", record=key, n=len(vals))
        npy = Path(vals[0]); npy = npy if npy.is_absolute() else DATASET_DIR / npy
    else:
        npy = find_npy(split, bearing, g["_index_file"].iloc[0])
    shape = np.load(npy, mmap_mode="r").shape
    rows = g[c_row].astype(int).to_numpy() if c_row else np.arange(len(g))
    if len(shape) != 2 or shape[1] != NS or rows.max() >= shape[0] or (not c_row and shape[0] != len(g)):
        stop("M2ENC_NPY_SHAPE_MISMATCH", record=key, npy_shape=shape, n_index=len(g))
    role = {"FULL_TEST": "TRAIN", "LEARNING": "EVAL", "TEST": "EXCLUDED"}[split]
    rec_rows.append(dict(record_key=key, split=split, bearing=bearing, n=len(g), role=role,
                         fold=FOLDS.get(bearing) if role == "TRAIN" else None,
                         npy_path=str(npy), npy_rows=shape[0],
                         note="FULL_TEST 앞부분 중복/#9 보류(D-026)" if role == "EXCLUDED" else ""))
    WIN[key] = rows
RECORDS = pd.DataFrame(rec_rows)

# 범위 검사
chk = {}
for sp, (n_exp, r_exp) in EXPECTED.items():
    sub = RECORDS[RECORDS.split == sp]
    chk[sp] = (int(sub.n.sum()), len(sub))
    if chk[sp] != (n_exp, r_exp): stop("M2ENC_SCOPE_MISMATCH", got=chk, expected=EXPECTED)
tr_b = set(RECORDS[RECORDS.role == "TRAIN"].bearing)
ev_b = set(RECORDS[RECORDS.role == "EVAL"].bearing)
if tr_b & ev_b: stop("M2ENC_TRAIN_EVAL_OVERLAP", overlap=sorted(tr_b & ev_b))
if tr_b != set(FOLDS): stop("M2ENC_FOLD_TABLE_MISMATCH", train=sorted(tr_b), folds=sorted(FOLDS))

RUN_DIR = OUT_ROOT / (datetime.datetime.now().strftime("%Y%m%d_%H%M%S") + "_m2_enc_v1_data_prep")
RUN_DIR.mkdir(parents=True, exist_ok=False)
mp = RUN_DIR / "split_manifest.csv"
RECORDS.to_csv(mp, index=False, encoding="utf-8-sig")
SPLIT_SHA = hashlib.sha256(mp.read_bytes()).hexdigest()

print("[SCOPE]", chk)
print(RECORDS[["record_key","role","fold","n"]].to_string(index=False))
print(f"\n[SPLIT] TRAIN {int(RECORDS[RECORDS.role=='TRAIN'].n.sum())}윈도 / "
      f"EVAL {int(RECORDS[RECORDS.role=='EVAL'].n.sum())} / EXCLUDED {int(RECORDS[RECORDS.role=='EXCLUDED'].n.sum())}")
print("[SPLIT] 학습∩평가 베어링 = 없음 ✅")
print("[OUT]", RUN_DIR, "\n[SHA256] split_manifest.csv", SPLIT_SHA)


[SCOPE] {'LEARNING': (7534, 6), 'TEST': (14127, 11), 'FULL_TEST': (17355, 11)}
          record_key     role  fold    n
FULL_TEST/Bearing1_3    TRAIN   0.0 2375
FULL_TEST/Bearing1_4    TRAIN   1.0 1428
FULL_TEST/Bearing1_5    TRAIN   2.0 2463
FULL_TEST/Bearing1_6    TRAIN   3.0 2448
FULL_TEST/Bearing1_7    TRAIN   4.0 2259
FULL_TEST/Bearing2_3    TRAIN   0.0 1955
FULL_TEST/Bearing2_4    TRAIN   1.0  751
FULL_TEST/Bearing2_5    TRAIN   2.0 2311
FULL_TEST/Bearing2_6    TRAIN   3.0  701
FULL_TEST/Bearing2_7    TRAIN   4.0  230
FULL_TEST/Bearing3_3    TRAIN   2.0  434
 LEARNING/Bearing1_1     EVAL   NaN 2803
 LEARNING/Bearing1_2     EVAL   NaN  871
 LEARNING/Bearing2_1     EVAL   NaN  911
 LEARNING/Bearing2_2     EVAL   NaN  797
 LEARNING/Bearing3_1     EVAL   NaN  515
 LEARNING/Bearing3_2     EVAL   NaN 1637
     TEST/Bearing1_3 EXCLUDED   NaN  590
     TEST/Bearing1_4 EXCLUDED   NaN 1139
     TEST/Bearing1_5 EXCLUDED   NaN 2302
     TEST/Bearing1_6 EXCLUDED   NaN 2232
     TEST/Bearing1_

In [ ]:
# ===== M2_ENC_v1 Cell 3: LP6000 + 스케일 s(D-027) + INT8 범위 Q 후보 측정 (학습 레코드만) =====
from scipy import signal

# 필터 재설계 + Next-4 수치와 대조 (게이트)
numtaps, beta = signal.kaiserord(60.0, 400.0 / (FS / 2))
if numtaps % 2 == 0: numtaps += 1
H_LP = signal.firwin(numtaps, 6000.0, window=("kaiser", beta), fs=FS)
_, Hf = signal.freqz(H_LP, worN=[5800.0, 6000.0, 6200.0], fs=FS)
resp = 20 * np.log10(np.abs(Hf))
filt_ok = (numtaps == 187 and abs(beta - 5.653) < 1e-3 and abs(resp[0] + 0.01) < 0.05
           and abs(resp[1] + 6.02) < 0.05 and resp[2] <= -60.0)
key_try = hashlib.sha256(H_LP.astype(np.float64).tobytes()).hexdigest()[:12]
print(f"[FILT] taps={numtaps} beta={beta:.3f} 응답 5800/6000/6200Hz = {resp.round(2).tolist()} dB")
print(f"[FILT] 계수 해시={key_try} (Next-4 FILT_KEY 7e32cc3e140c와 산출식이 다를 수 있음, 참고용)")
if not filt_ok: stop("M2ENC_FILTER_MISMATCH", taps=numtaps, beta=beta, resp=resp.tolist())
PAD = (numtaps - 1) // 2

def lp6000(X):
    Xp = np.pad(X.astype(np.float64), ((0, 0), (PAD, PAD)), mode="reflect")
    return signal.fftconvolve(Xp, H_LP[None, :], mode="valid", axes=1)

def load_record(r, tries=5):
    for t in range(tries):
        try:
            A = np.load(r.npy_path, mmap_mode="r")
            return np.asarray(A[WIN[r.record_key]], dtype=np.float32)
        except OSError as e:
            print(f"   재시도 {t+1}/{tries}: {r.record_key} {e}"); time.sleep(5 * (t + 1))
    stop("M2ENC_READ_FAILED", record=r.record_key)

rows, all_pool, norm_pool, tail_pool = [], [], [], []
for r in RECORDS[RECORDS.role == "TRAIN"].itertuples():
    X = load_record(r)
    if X.shape != (r.n, NS) or not np.isfinite(X).all():
        stop("M2ENC_BAD_SIGNAL", record=r.record_key, shape=X.shape)
    Y = lp6000(X)
    rms = np.sqrt((Y ** 2).mean(axis=1))
    s = float(np.median(rms[:REG_MIN]))
    Z = np.abs(Y / s).astype(np.float32)
    nt = max(1, int(round(0.1 * r.n)))
    all_pool.append(Z.ravel()); norm_pool.append(Z[:REG_MIN].ravel()); tail_pool.append(Z[-nt:].ravel())
    rows.append(dict(record_key=r.record_key, fold=r.fold, n=r.n, s=s,
                     dc_ratio=float(np.abs(Y[:REG_MIN].mean()) / s),
                     tail_rms_ratio=float(np.median(rms[-nt:]) / s),
                     absz_p999_normal=float(np.percentile(Z[:REG_MIN], 99.9)),
                     absz_max=float(Z.max())))
    print(f"  {r.record_key:22s} n={r.n:5d} s={s:.5f} 말기RMS/s={rows[-1]['tail_rms_ratio']:.2f}")
    del X, Y, Z

SCALE = pd.DataFrame(rows)
A, N_, T_ = (np.concatenate(p) for p in (all_pool, norm_pool, tail_pool))
qc = {"ALL_p99.9": np.percentile(A, 99.9), "ALL_p99.99": np.percentile(A, 99.99), "ALL_max": A.max(),
      "NORMAL_p99.9": np.percentile(N_, 99.9), "NORMAL_max": N_.max()}
n999 = np.percentile(N_, 99.9)
QTAB = pd.DataFrame([dict(rule=k, Q=float(q),
                          normal_levels_used=int(2 * min(127, round(127 * n999 / q)) + 1),
                          clip_rate_normal=float((N_ > q).mean()),
                          clip_rate_tail10=float((T_ > q).mean()),
                          clip_rate_all=float((A > q).mean())) for k, q in qc.items()])

SCALE.to_csv(RUN_DIR / "scale_by_record.csv", index=False, encoding="utf-8-sig")
QTAB.to_csv(RUN_DIR / "q_candidates.csv", index=False, encoding="utf-8-sig")
summary = dict(status="M2ENC_DATA_PREP_MEASURED_AWAITING_Q_RULE", contract=CONTRACT_VERSION,
               run_dir=str(RUN_DIR), split_manifest_sha256=SPLIT_SHA,
               filter=dict(taps=int(numtaps), beta=float(beta), resp_db=resp.round(2).tolist(),
                           coef_hash_try=key_try),
               train_windows=int(SCALE.n.sum()), learning_signals_read=False,
               baseline_d025=BASELINE_D025, criteria_d028=CRITERIA_D028)
(RUN_DIR / "DATA_PREP_SUMMARY.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

pd.set_option("display.width", 200)
print("\n[SCALE]\n", SCALE.round(4).to_string(index=False))
print("\n[Q 후보] (정상 사용 단계 수: 256단계 중 정상 구간이 실제로 쓰는 단계)\n", QTAB.round(5).to_string(index=False))
print("\nSTATUS:", summary["status"], "| LEARNING 신호 읽기: 없음")


[FILT] taps=187 beta=5.653 응답 5800/6000/6200Hz = [-0.01, -6.02, -60.96] dB
[FILT] 계수 해시=b5543b5a8f89 (Next-4 FILT_KEY 7e32cc3e140c와 산출식이 다를 수 있음, 참고용)
  FULL_TEST/Bearing1_3   n= 2375 s=0.36424 말기RMS/s=4.87
  FULL_TEST/Bearing1_4   n= 1428 s=0.39174 말기RMS/s=14.77
  FULL_TEST/Bearing1_5   n= 2463 s=0.32944 말기RMS/s=0.72
  FULL_TEST/Bearing1_6   n= 2448 s=0.41906 말기RMS/s=0.70
  FULL_TEST/Bearing1_7   n= 2259 s=0.42610 말기RMS/s=0.83
  FULL_TEST/Bearing2_3   n= 1955 s=0.47478 말기RMS/s=0.50
  FULL_TEST/Bearing2_4   n=  751 s=0.34546 말기RMS/s=0.77
  FULL_TEST/Bearing2_5   n= 2311 s=0.65251 말기RMS/s=0.36
  FULL_TEST/Bearing2_6   n=  701 s=0.28748 말기RMS/s=0.69
  FULL_TEST/Bearing2_7   n=  230 s=0.33397 말기RMS/s=1.14
  FULL_TEST/Bearing3_3   n=  434 s=0.25796 말기RMS/s=0.84

[SCALE]
           record_key  fold    n      s  dc_ratio  tail_rms_ratio  absz_p999_normal  absz_max
FULL_TEST/Bearing1_3   0.0 2375 0.3642    0.0018          4.8670            3.7022  149.4143
FULL_TEST/Bearing1_4   1.0 1428 0.39

In [ ]:
# ===== M2_ENC_v1 Cell 4: output_row_index 전수 검증 + 학습 데이터 적재 (LP6000 → x/s → clip ±Q) =====
Q_IN = 8.0                       # D-029
EXPECTED_SPLIT_SHA = "41a0e1f243eb1fa98b4d8dfc894b281ab843e6faf7f9d5902b7f0a97d790c10e"
if SPLIT_SHA != EXPECTED_SPLIT_SHA:
    stop("M2ENC_SPLIT_CHANGED", got=SPLIT_SHA, expected=EXPECTED_SPLIT_SHA)

for c in ["output_array_relative_path", "output_row_index"]:
    if c not in IDX.columns: stop("M2ENC_COLUMN_NOT_FOUND", what=c, available=list(IDX.columns))

map_issues = []
for r in RECORDS.itertuples():
    g = IDX[IDX._record_key == r.record_key].sort_values("_seq")
    seq = g._seq.to_numpy()
    rows = g.output_row_index.astype(int).to_numpy()
    paths = g.output_array_relative_path.astype(str).str.strip().unique()
    p_ok = len(paths) == 1 and os.path.normpath(str(DATASET_DIR / paths[0])) == os.path.normpath(r.npy_path)
    if not p_ok or not np.all(np.diff(seq) > 0) or rows.min() < 0 or rows.max() >= r.npy_rows:
        stop("M2ENC_ROW_MAPPING_INVALID", record=r.record_key, paths=list(paths)[:3])
    if not np.array_equal(rows, WIN[r.record_key]):
        map_issues.append(r.record_key)
    WIN[r.record_key] = rows          # 이후는 output_row_index 기준
print(f"[MAP] 28개 레코드 검증 완료. Cell 2 가정(행순서=sequence)과 다른 레코드: {map_issues or '없음 ✅'}")
if map_issues:
    print("   ⚠ Cell 3의 s 값은 잘못된 행 순서로 계산됨 → 아래 s 재계산값을 기준으로 사용")

DATA, chk_rows = {}, []
for r in RECORDS[RECORDS.role == "TRAIN"].itertuples():
    X = load_record(r)
    if X.shape != (r.n, NS) or not np.isfinite(X).all():
        stop("M2ENC_BAD_SIGNAL", record=r.record_key, shape=X.shape)
    Y = lp6000(X)
    rms = np.sqrt((Y ** 2).mean(axis=1))
    s = float(np.median(rms[:REG_MIN]))
    Z = np.clip(Y / s, -Q_IN, Q_IN).astype(np.float32)
    DATA[r.record_key] = dict(Z=Z, t=np.arange(r.n) / (r.n - 1), s=s, fold=int(r.fold), n=r.n)
    s3 = float(SCALE.loc[SCALE.record_key == r.record_key, "s"].iloc[0])
    chk_rows.append(dict(record_key=r.record_key, fold=int(r.fold), n=r.n, s=s,
                         s_cell3=s3, s_match=abs(s - s3) <= 1e-9 * max(1.0, abs(s3)),
                         clip_rate=float((np.abs(Y / s) > Q_IN).mean())))
    del X, Y
CHK = pd.DataFrame(chk_rows)
print(CHK.round(6).to_string(index=False))
print(f"\n[DATA] 학습 윈도 {sum(v['n'] for v in DATA.values())}개, 메모리 약 "
      f"{sum(v['Z'].nbytes for v in DATA.values())/1e6:.0f}MB, 전체 clip 비율 "
      f"{np.average(CHK.clip_rate, weights=CHK.n):.5f}")
if not map_issues and not CHK.s_match.all():
    stop("M2ENC_SCALE_NOT_REPRODUCED", bad=CHK[~CHK.s_match].record_key.tolist())
print("Cell 4 완료")


[MAP] 28개 레코드 검증 완료. Cell 2 가정(행순서=sequence)과 다른 레코드: 없음 ✅
          record_key  fold    n        s  s_cell3  s_match  clip_rate
FULL_TEST/Bearing1_3     0 2375 0.364240 0.364240     True   0.017466
FULL_TEST/Bearing1_4     1 1428 0.391743 0.391743     True   0.090027
FULL_TEST/Bearing1_5     2 2463 0.329442 0.329442     True   0.000244
FULL_TEST/Bearing1_6     3 2448 0.419063 0.419063     True   0.000152
FULL_TEST/Bearing1_7     4 2259 0.426104 0.426104     True   0.000861
FULL_TEST/Bearing2_3     0 1955 0.474782 0.474782     True   0.001919
FULL_TEST/Bearing2_4     1  751 0.345464 0.345464     True   0.000373
FULL_TEST/Bearing2_5     2 2311 0.652514 0.652514     True   0.000004
FULL_TEST/Bearing2_6     3  701 0.287482 0.287482     True   0.000782
FULL_TEST/Bearing2_7     4  230 0.333969 0.333969     True   0.012511
FULL_TEST/Bearing3_3     2  434 0.257961 0.257961     True   0.001115

[DATA] 학습 윈도 17355개, 메모리 약 142MB, 전체 clip 비율 0.01042
Cell 4 완료


In [ ]:
# ===== M2_ENC_v1 Cell 5 (r2): 동결 M0 판정 함수 추출(재구현 금지) + 임베딩용 어댑터 =====
# [r2] 동결 노트북의 실행용 함수(run_m0 등)가 미정의 기본값(FEMTO_ROOT)을 참조해 NameError →
#      노드 단위 실행 실패는 기록 후 계속(COMPAT Cell 2와 동일 방식).
#      대신 register_and_detect가 참조하는 이름이 하나라도 미정의면 정지(안전장치).
#      판정 로직·상수 변경 없음.
import ast, inspect, json, hashlib, dis, builtins
import numpy as np
import pandas as pd

FROZEN_NB = PROJECT_ROOT / "M0_FEMTO_Baseline_v1.ipynb"
FROZEN_SHA_PREFIX = "921f1bbea7b460d748c6dc3f74c04981f3bf4de5ff76223b"
FROZEN_DETECT_FN = "register_and_detect"
EXPECTED_CONSTS = {"K": 6.0, "CONSEC": 5, "REG_MIN": 90, "REG_MAX": 600}

raw = FROZEN_NB.read_bytes()
nb_sha = hashlib.sha256(raw).hexdigest()
if not nb_sha.startswith(FROZEN_SHA_PREFIX):
    stop("M2ENC_FROZEN_NOTEBOOK_CHANGED", sha=nb_sha)

ns, skipped, bad_cells, exec_fail = {"__name__": "frozen_m0"}, [], 0, []
for ci, cell in enumerate(json.loads(raw.decode("utf-8"))["cells"]):
    if cell.get("cell_type") != "code": continue
    src = "".join(cell["source"]) if isinstance(cell["source"], list) else cell["source"]
    src = "\n".join(l for l in src.splitlines() if not l.lstrip().startswith(("!", "%")))
    try: tree = ast.parse(src)
    except SyntaxError: bad_cells += 1; continue
    for node in tree.body:
        keep = isinstance(node, (ast.Import, ast.ImportFrom, ast.FunctionDef, ast.ClassDef))
        if isinstance(node, ast.Assign) and all(isinstance(t, ast.Name) and t.id.isupper() for t in node.targets):
            try: ast.literal_eval(node.value); keep = True
            except Exception: skipped.append((ast.get_source_segment(src, node) or "")[:80])
        if keep:
            try:
                exec(compile(ast.Module([node], []), f"<frozen cell {ci}>", "exec"), ns)
            except Exception as e:   # [r2] 기록 후 계속
                name = getattr(node, "name", None) or (ast.get_source_segment(src, node) or "")[:60]
                exec_fail.append({"cell": ci, "name": name, "error": repr(e)[:120]})

if FROZEN_DETECT_FN not in ns:
    stop("M2ENC_FROZEN_FN_NOT_FOUND", available=sorted(k for k in ns if callable(ns[k]))[:50])
bad = {k: ns.get(k) for k, v in EXPECTED_CONSTS.items() if ns.get(k) != v}
if bad: stop("M2ENC_FROZEN_CONST_MISMATCH", got=bad, expected=EXPECTED_CONSTS)
FN_DET = ns[FROZEN_DETECT_FN]

# [r2] 안전장치: 판정 함수(및 그 안에서 부르는 동결 함수)가 참조하는 이름이 모두 정의돼 있는지 확인
def _names(code):
    out = {i.argval for i in dis.get_instructions(code) if i.opname in ("LOAD_GLOBAL", "LOAD_NAME")}
    for c in code.co_consts:
        if hasattr(c, "co_code"): out |= _names(c)
    return {str(n) for n in out}

def _unresolved(fn, seen=None):
    seen = set() if seen is None else seen
    if fn.__name__ in seen: return set()
    seen.add(fn.__name__); miss = set()
    for n in _names(fn.__code__):
        if n in ns:
            o = ns[n]
            if inspect.isfunction(o) and o.__globals__ is ns: miss |= _unresolved(o, seen)
        elif not hasattr(builtins, n):
            miss.add(n)
    return miss

_miss = sorted(_unresolved(FN_DET))
if _miss:
    stop("M2ENC_FROZEN_FN_UNRESOLVED", missing=_miss, exec_fail=exec_fail)

print(f"[FROZEN] sha={nb_sha[:16]}… , 파싱 불가 셀 {bad_cells}개, 실행 제외 대입문 {len(skipped)}개, 실행 실패 {len(exec_fail)}건")
if exec_fail:
    print("[FROZEN] 실행 실패 목록(판정 함수와 무관하면 정상 — 예: run_m0):")
    display(pd.DataFrame(exec_fail))
print(f"[FROZEN] {FROZEN_DETECT_FN}{inspect.signature(FN_DET)}  상수 {EXPECTED_CONSTS} ✅  미해결 참조 0건")

ONSET_KEYS  = ["onset", "onset_idx", "alarm_idx", "alarm_index", "first_alarm_idx"]
FAR_KEYS    = ["reg_far", "reg_FAR", "reg_far_ratio", "far_reg"]
STATUS_KEYS = ["status", "detected", "is_detected", "result"]

def _get(d, keys):
    for k in keys:
        if k in d: return k, d[k]
    return None, None

def _status_detected(v):
    if isinstance(v, (bool, np.bool_)): return bool(v)
    s = str(v); sl = s.lower()
    if "미탐지" in s or "not" in sl or "miss" in sl: return False
    return ("탐지" in s) or ("detect" in sl)

def run_detector(E):
    out = FN_DET(np.asarray(E, dtype=np.float64))
    if not isinstance(out, dict):
        stop("M2ENC_DETECTOR_OUTPUT_NOT_DICT", type=str(type(out)), repr=repr(out)[:800])
    ko, onset = _get(out, ONSET_KEYS); kf, far = _get(out, FAR_KEYS); ks, st = _get(out, STATUS_KEYS)
    if kf is None or (ko is None and ks is None):
        stop("M2ENC_DETECTOR_KEYS_UNKNOWN", keys=list(out),
             sample={k: (v if np.isscalar(v) else type(v).__name__) for k, v in out.items()})
    det = onset is not None and not (isinstance(onset, float) and np.isnan(onset)) and int(onset) >= 0
    if ks is not None and _status_detected(st) != det:
        stop("M2ENC_DETECTOR_STATUS_MISMATCH", status=str(st), onset=str(onset))
    return dict(detected=bool(det), onset=int(onset) if det else None, reg_far=float(far))

# 스모크 테스트 (합성 데이터, 판정 기준 확인용 아님)
rng = np.random.default_rng(0)
E_syn = rng.normal(size=(600, 8)); E_syn[450:] += np.linspace(0, 8, 150)[:, None]
try:
    raw_out = FN_DET(E_syn)
except TypeError as e:
    stop("M2ENC_DETECTOR_SIGNATURE_UNKNOWN", sig=str(inspect.signature(FN_DET)), err=str(e))
print("[SMOKE] 원출력 키:", list(raw_out) if isinstance(raw_out, dict) else type(raw_out))
res = run_detector(E_syn)
print("[SMOKE] 어댑터 결과:", res, "" if res["detected"] else "※ 합성 변화 미검출(보고만)")
print("Cell 5 완료 (r2)")



[FROZEN] sha=921f1bbea7b460d7… , 파싱 불가 셀 0개, 실행 제외 대입문 3개, 실행 실패 1건
[FROZEN] 실행 실패 목록(판정 함수와 무관하면 정상 — 예: run_m0):


,cell,name,error
0,3,run_m0,"NameError(""name 'FEMTO_ROOT' is not defined"")"


[FROZEN] register_and_detect(feats, k=6.0, consec=5, reg_min=90, reg_max=600)  상수 {'K': 6.0, 'CONSEC': 5, 'REG_MIN': 90, 'REG_MAX': 600} ✅  미해결 참조 0건
[SMOKE] 원출력 키: ['n_reg', 'thr', 'dists', 'alarm_idx', 'N', 'reg_far']
[SMOKE] 어댑터 결과: {'detected': True, 'onset': 499, 'reg_far': 0.0} 
Cell 5 완료 (r2)


In [ ]:
# ===== M2_ENC_v1 Cell 6: 4구성 × 5-fold 학습 (D-030), 체크포인트로 이어하기 =====
import math, torch, torch.nn as nn
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("[DEVICE]", DEV, "" if DEV == "cuda" else "⚠ CPU: 매우 느림, GPU 런타임 권장")

HP = dict(epochs=30, batch=256, lr=1e-3, wd=1e-4, lam_compact=0.1,
          normal_frac=0.3, knee=0.7, seed=20261030, aug="roll+signflip", arch="v1a")
CONFIGS = [("LIN", 8), ("LIN", 16), ("KNEE", 8), ("KNEE", 16)]
TRAIN_KEY = hashlib.sha256(json.dumps(dict(split=SPLIT_SHA, Q=Q_IN, hp=HP, cfg=CONFIGS),
                                      sort_keys=True).encode()).hexdigest()[:12]
TRAIN_DIR = OUT_ROOT / f"m2_enc_v1_train_{TRAIN_KEY}"
(TRAIN_DIR / "ckpt").mkdir(parents=True, exist_ok=True)

def blk(ci, co, k, s, p):
    return nn.Sequential(nn.Conv1d(ci, co, k, s, p, bias=False), nn.BatchNorm1d(co), nn.ReLU())
class Enc(nn.Module):
    def __init__(self, D):
        super().__init__()
        self.f = nn.Sequential(blk(1, 4, 12, 4, 4), blk(4, 8, 12, 4, 4), blk(8, 16, 6, 2, 2),
                               blk(16, 32, 6, 2, 2), blk(32, 32, 3, 1, 1),
                               nn.AdaptiveAvgPool1d(1), nn.Flatten())
        self.emb = nn.Linear(32, D)
    def forward(self, x): return self.emb(self.f(x))
class Net(nn.Module):
    def __init__(self, D):
        super().__init__(); self.enc = Enc(D); self.head = nn.Linear(D, 1)

def target(t, mode):
    return t if mode == "LIN" else np.clip((t - HP["knee"]) / (1 - HP["knee"]), 0, 1)

def fold_tensors(fold_out, mode):
    keys = [k for k, v in DATA.items() if v["fold"] != fold_out]
    cat = lambda f: np.concatenate([f(i, k) for i, k in enumerate(keys)])
    Z = cat(lambda i, k: DATA[k]["Z"])
    y = cat(lambda i, k: target(DATA[k]["t"], mode))
    rid = cat(lambda i, k: np.full(DATA[k]["n"], i))
    w = cat(lambda i, k: np.full(DATA[k]["n"], 1.0 / DATA[k]["n"])); w = w / w.mean()
    nm = cat(lambda i, k: DATA[k]["t"] < HP["normal_frac"])
    T = lambda a, dt: torch.as_tensor(a, dtype=dt, device=DEV)
    return T(Z, torch.float32), T(y, torch.float32), T(rid, torch.long), T(w, torch.float32), T(nm, torch.bool)

def compact_loss(e, r, m):
    e2, r2 = e[m], r[m]; parts = []
    for u in torch.unique(r2):
        eu = e2[r2 == u]
        if len(eu) >= 2: parts.append(((eu - eu.mean(0)) ** 2).sum(1).mean())
    if not parts: return e.new_zeros(())
    return torch.stack(parts).mean() / (e.var(0).sum() + 1e-6)   # 스케일 불변(붕괴 방지)

def train_one(mode, D, fold_out, seed):
    torch.manual_seed(seed); g = torch.Generator().manual_seed(seed)
    Zt, yt, rt, wt, nt = fold_tensors(fold_out, mode); N = len(yt)
    net = Net(D).to(DEV)
    opt = torch.optim.AdamW(net.parameters(), lr=HP["lr"], weight_decay=HP["wd"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=HP["epochs"] * math.ceil(N / HP["batch"]))
    hist = []
    for ep in range(HP["epochs"]):
        net.train(); perm = torch.randperm(N, generator=g).to(DEV); acc = [0.0, 0.0, 0]
        for i in range(0, N, HP["batch"]):
            idx = perm[i:i + HP["batch"]]
            x = torch.roll(Zt[idx].unsqueeze(1), int(torch.randint(0, NS, (1,), generator=g)), dims=-1)
            x = x * (torch.randint(0, 2, (len(idx), 1, 1), generator=g).float() * 2 - 1).to(DEV)
            e = net.enc(x); p = net.head(e).squeeze(-1)
            l_r = (wt[idx] * (p - yt[idx]) ** 2).mean(); l_c = compact_loss(e, rt[idx], nt[idx])
            loss = l_r + HP["lam_compact"] * l_c
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            acc[0] += l_r.item(); acc[1] += float(l_c); acc[2] += 1
        hist.append((ep, acc[0] / acc[2], acc[1] / acc[2]))
    return net, hist

n_params = sum(p.numel() for p in Enc(16).parameters())
print(f"[MODEL] 인코더 파라미터 {n_params}개 (D=16 기준), TRAIN_DIR={TRAIN_DIR}")
(TRAIN_DIR / "train_config.json").write_text(json.dumps(dict(train_key=TRAIN_KEY, hp=HP, configs=CONFIGS,
    Q_in=Q_IN, split_sha=SPLIT_SHA, folds=FOLDS, n_params_D16=n_params), ensure_ascii=False, indent=2), encoding="utf-8")

t0 = time.time()
for ci, (mode, D) in enumerate(CONFIGS):
    for f in range(5):
        ck = TRAIN_DIR / "ckpt" / f"{mode}_D{D}_f{f}.pt"
        if ck.exists(): print(f"  RESUME SKIP {ck.name}"); continue
        ts = time.time()
        net, hist = train_one(mode, D, f, HP["seed"] + 100 * ci + f)
        tmp = ck.with_suffix(".tmp")
        torch.save(dict(state=net.state_dict(), mode=mode, D=D, fold=f, hist=hist, hp=HP), tmp)
        os.replace(tmp, ck)
        print(f"  {ck.name}: 회귀손실 {hist[-1][1]:.4f}, 응집손실 {hist[-1][2]:.4f}, {time.time()-ts:.0f}s")
print(f"Cell 6 완료 ({(time.time()-t0)/60:.1f}분)")


[DEVICE] cuda 
[MODEL] 인코더 파라미터 8056개 (D=16 기준), TRAIN_DIR=/content/drive/MyDrive/Colab Notebooks/field_iis3dwb/bridge_outputs/m2_enc_v1_train_42f861b9506e
  RESUME SKIP LIN_D8_f0.pt
  RESUME SKIP LIN_D8_f1.pt
  RESUME SKIP LIN_D8_f2.pt
  RESUME SKIP LIN_D8_f3.pt
  RESUME SKIP LIN_D8_f4.pt
  RESUME SKIP LIN_D16_f0.pt
  RESUME SKIP LIN_D16_f1.pt
  RESUME SKIP LIN_D16_f2.pt
  RESUME SKIP LIN_D16_f3.pt
  RESUME SKIP LIN_D16_f4.pt
  RESUME SKIP KNEE_D8_f0.pt
  RESUME SKIP KNEE_D8_f1.pt
  RESUME SKIP KNEE_D8_f2.pt
  RESUME SKIP KNEE_D8_f3.pt
  RESUME SKIP KNEE_D8_f4.pt
  RESUME SKIP KNEE_D16_f0.pt
  RESUME SKIP KNEE_D16_f1.pt
  RESUME SKIP KNEE_D16_f2.pt
  RESUME SKIP KNEE_D16_f3.pt
  RESUME SKIP KNEE_D16_f4.pt
Cell 6 완료 (0.0분)


In [ ]:
# ===== M2_ENC_v1 Cell 7: fold 검증(동결 판정) + D-030 선택 규칙. LEARNING 미사용 =====
@torch.no_grad()
def embed(net, Z, bs=512):
    net.eval(); out = []
    for i in range(0, len(Z), bs):
        out.append(net.enc(torch.from_numpy(Z[i:i + bs]).unsqueeze(1).to(DEV)).double().cpu().numpy())
    return np.concatenate(out)

rows = []
for mode, D in CONFIGS:
    for f in range(5):
        ck = torch.load(TRAIN_DIR / "ckpt" / f"{mode}_D{D}_f{f}.pt", map_location=DEV)
        net = Net(D).to(DEV); net.load_state_dict(ck["state"])
        for k, v in DATA.items():
            if v["fold"] != f: continue
            E = embed(net, v["Z"]); res = run_detector(E)
            lead = (v["n"] - res["onset"]) * 10 / 3600 if res["detected"] else np.nan
            rows.append(dict(config=f"{mode}_D{D}", mode=mode, D=D, fold=f, record_key=k, n=v["n"],
                             detected=res["detected"], onset=res["onset"], lead_h=lead,
                             reg_far_pct=100 * res["reg_far"], emb_min_std=float(E[:REG_MIN].std(0).min())))
VAL = pd.DataFrame(rows)
VAL.to_csv(TRAIN_DIR / "fold_validation.csv", index=False, encoding="utf-8-sig")

AGG = VAL.groupby(["config", "mode", "D"]).agg(detected=("detected", "sum"), n_rec=("record_key", "count"),
        mean_lead_h=("lead_h", "mean"), mean_reg_far_pct=("reg_far_pct", "mean"),
        min_emb_std=("emb_min_std", "min")).reset_index()
best_det = AGG.detected.max()
cand = AGG[(AGG.detected == best_det) & (AGG.mean_reg_far_pct <= CRITERIA_D028["mean_reg_far_pct_max"])]
if cand.empty:
    status, sel = "M2ENC_SELECTION_NO_CANDIDATE", None
else:
    sel = cand.sort_values(["mean_lead_h", "D"], ascending=[False, True]).iloc[0]
    status = "M2ENC_CV_SELECTED_AWAITING_FINAL"

summary = dict(status=status, train_key=TRAIN_KEY, train_dir=str(TRAIN_DIR),
               selected=None if sel is None else dict(config=sel.config, mode=sel["mode"], D=int(sel.D)),
               rule="max detected → mean_reg_far≤0.924% → max mean_lead → smaller D",
               learning_evaluated=False, unresolved_12_open=True,
               aggregate=AGG.round(4).to_dict(orient="records"))
(TRAIN_DIR / "CV_SELECTION_SUMMARY.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

pd.set_option("display.width", 200)
print(VAL.pivot_table(index="record_key", columns="config", values="onset", aggfunc="first").to_string())
print("\n[AGG] (11개 학습 레코드, 각 레코드는 자기 fold 모델로 1회 검증)\n", AGG.round(4).to_string(index=False))
print("\nSTATUS:", status, "| 선택:", None if sel is None else sel.config, "| LEARNING 평가: 안 함")


config                KNEE_D16  KNEE_D8  LIN_D16  LIN_D8
record_key                                              
FULL_TEST/Bearing1_3    1117.0   1254.0   1319.0  1219.0
FULL_TEST/Bearing1_4     756.0    880.0    653.0   692.0
FULL_TEST/Bearing1_5    1276.0   1575.0    920.0  1101.0
FULL_TEST/Bearing1_6     841.0   1591.0   1234.0  1587.0
FULL_TEST/Bearing1_7    1741.0    959.0   1169.0  2211.0
FULL_TEST/Bearing2_3    1944.0   1944.0   1929.0  1937.0
FULL_TEST/Bearing2_4     397.0    485.0    397.0   492.0
FULL_TEST/Bearing2_6     350.0    411.0    355.0   428.0
FULL_TEST/Bearing2_7     206.0    223.0    176.0   182.0
FULL_TEST/Bearing3_3     423.0    423.0    249.0   324.0

[AGG] (11개 학습 레코드, 각 레코드는 자기 fold 모델로 1회 검증)
   config mode  D  detected  n_rec  mean_lead_h  mean_reg_far_pct  min_emb_std
KNEE_D16 KNEE 16        10     11       1.6647            0.0933       0.0072
 KNEE_D8 KNEE  8        10     11       1.4719            0.1711       0.0135
 LIN_D16  LIN 16        10     11  

In [ ]:
# ===== M2_ENC_v1 Cell 7B (r2): 진단 전용 — 같은 FULL_TEST 11개에서 M0-PHY(LP6000) vs M2-ENC(선택 구성) =====
# [r2] 동결 extract_features(sig, fs=25600) → 실제 입력 fs(20480) 명시 전달 + fs 민감도 자동 확인.
# D-030 선택 규칙·결과 변경 없음. LEARNING 미사용. 쓰기: TRAIN_DIR 안 진단 파일 2개만.
FROZEN_FEATURE_FN = "extract_features"
if sel is None: stop("M2ENC_DIAG_NO_SELECTION")
if FROZEN_FEATURE_FN not in ns:
    stop("M2ENC_FROZEN_FEATURE_FN_NOT_FOUND", available=sorted(k for k in ns if callable(ns[k]))[:50])
FN_FEAT = ns[FROZEN_FEATURE_FN]
sig = inspect.signature(FN_FEAT)
print("[M0] 동결 특징 함수:", FROZEN_FEATURE_FN, sig)

FS_PARAM = next((p for p in sig.parameters if p.lower() == "fs"), None)
FEAT_KW = {FS_PARAM: FS} if FS_PARAM else {}          # FS = 20480 (Cell 1)

def _feat_raw(x, **kw):
    out = FN_FEAT(np.asarray(x, dtype=np.float64), **kw)
    return np.asarray(list(out.values()) if isinstance(out, dict) else out, dtype=np.float64).ravel()

def feat9(x): return _feat_raw(x, **FEAT_KW)

# fs 민감도 확인: 같은 신호를 fs=20480 / fs=25600으로 계산해 비교
_x = lp6000(np.random.default_rng(1).normal(size=(1, NS)))[0]
try:
    _v = feat9(_x)
    _v_25k = _feat_raw(_x, **{FS_PARAM: 25600}) if FS_PARAM else _v
except Exception as e:
    stop("M2ENC_FEATURE_CALL_FAILED", sig=str(sig), err=repr(e)[:300])
if _v.size != 9: stop("M2ENC_FEATURE_DIM_UNEXPECTED", dim=int(_v.size), sig=str(sig))
FS_SENSITIVE = not np.allclose(_v, _v_25k, rtol=1e-12, atol=0.0)
_diff_idx = [int(i) for i in np.where(~np.isclose(_v, _v_25k, rtol=1e-12, atol=0.0))[0]]
print(f"[M0] fs 전달: {FEAT_KW or '없음'} | fs 민감도: "
      + (f"있음 ⚠ (특징 인덱스 {_diff_idx}가 fs에 따라 달라짐 → D-025 기준선 fs 확인 필요 [확인 대기])"
         if FS_SENSITIVE else "없음 ✅ (9특징 모두 fs 무관 — 결과 그대로 유효)"))

def diag(F_or_E, n):
    raw = FN_DET(np.asarray(F_or_E, dtype=np.float64)); res = run_detector(F_or_E)
    d = np.asarray(raw.get("dists", []), dtype=np.float64); thr = float(raw.get("thr", np.nan))
    on = res["onset"]
    post = float((d[on:] > thr).mean()) if (res["detected"] and len(d) == n) else np.nan
    return dict(detected=res["detected"], onset=on,
                onset_frac=(on / n) if res["detected"] else np.nan,
                lead_h=((n - on) * 10 / 3600) if res["detected"] else np.nan,
                reg_far_pct=100 * res["reg_far"], post_exceed=post)

mode_s, D_s = sel["mode"], int(sel.D)
rows7b, t0 = [], time.time()
for r in RECORDS[RECORDS.role == "TRAIN"].itertuples():
    X = load_record(r); Y = lp6000(X)                      # M0: 2048 + LP6000 (클리핑 없음)
    F = np.stack([feat9(y) for y in Y]); del X, Y
    ck = torch.load(TRAIN_DIR / "ckpt" / f"{mode_s}_D{D_s}_f{int(r.fold)}.pt", map_location=DEV)
    net = Net(D_s).to(DEV); net.load_state_dict(ck["state"])
    E = embed(net, DATA[r.record_key]["Z"])
    for tag, M in (("M0_PHY", F), (f"M2_{mode_s}_D{D_s}", E)):
        rows7b.append(dict(record_key=r.record_key, model=tag, n=r.n,
                           tail_rms_ratio=float(SCALE.loc[SCALE.record_key == r.record_key, "tail_rms_ratio"].iloc[0]),
                           **diag(M, r.n)))
    print(f"  {r.record_key:22s} 완료 ({time.time()-t0:.0f}s)")

DIAG = pd.DataFrame(rows7b)
DIAG.to_csv(TRAIN_DIR / "cv_diag_m0_vs_m2.csv", index=False, encoding="utf-8-sig")
AGG7 = DIAG.groupby("model").agg(detected=("detected", "sum"), mean_lead_h=("lead_h", "mean"),
                                 mean_reg_far_pct=("reg_far_pct", "mean"),
                                 median_onset_frac=("onset_frac", "median"),
                                 mean_post_exceed=("post_exceed", "mean")).reset_index()
(TRAIN_DIR / "CV_DIAG_SUMMARY.json").write_text(json.dumps(dict(
    status="M2ENC_CV_DIAG_COMPLETE", selection_unchanged=True, selected=sel.config,
    feature_fn_sig=str(sig), feature_fs_passed=FEAT_KW, feature_fs_sensitive=bool(FS_SENSITIVE),
    feature_fs_sensitive_idx=_diff_idx,
    m0_baseline_fs_match="CONFIRM_PENDING" if FS_SENSITIVE else "NOT_APPLICABLE",
    learning_evaluated=False,
    aggregate=AGG7.round(4).to_dict(orient="records")), ensure_ascii=False, indent=2, default=str), encoding="utf-8")

pd.set_option("display.width", 220)
print("\n[레코드별]\n", DIAG.pivot_table(index=["record_key", "tail_rms_ratio"], columns="model",
      values=["onset_frac", "post_exceed"], aggfunc="first").round(3).to_string())
print("\n[요약]\n", AGG7.round(4).to_string(index=False))
print("\nSTATUS: M2ENC_CV_DIAG_COMPLETE | 선택 변경 없음 | LEARNING 평가: 안 함"
      + (" | ⚠ M0 fs 민감 → [확인 대기]" if FS_SENSITIVE else ""))



[M0] 동결 특징 함수: extract_features (sig, fs=25600)
[M0] fs 전달: {'fs': 20480} | fs 민감도: 있음 ⚠ (특징 인덱스 [8]가 fs에 따라 달라짐 → D-025 기준선 fs 확인 필요 [확인 대기])
  FULL_TEST/Bearing1_3   완료 (6s)
  FULL_TEST/Bearing1_4   완료 (9s)
  FULL_TEST/Bearing1_5   완료 (13s)
  FULL_TEST/Bearing1_6   완료 (18s)
  FULL_TEST/Bearing1_7   완료 (22s)
  FULL_TEST/Bearing2_3   완료 (25s)
  FULL_TEST/Bearing2_4   완료 (26s)
  FULL_TEST/Bearing2_5   완료 (30s)
  FULL_TEST/Bearing2_6   완료 (32s)
  FULL_TEST/Bearing2_7   완료 (32s)
  FULL_TEST/Bearing3_3   완료 (33s)

[레코드별]
                                     onset_frac            post_exceed           
model                                   M0_PHY M2_LIN_D16      M0_PHY M2_LIN_D16
record_key           tail_rms_ratio                                             
FULL_TEST/Bearing1_3 4.867040            0.560      0.555       0.987      0.973
FULL_TEST/Bearing1_4 14.774470           0.757      0.457       1.000      0.955
FULL_TEST/Bearing1_5 0.715158            0.978      0.374       1.000  

In [ ]:
# ===== M2_ENC_v1 Cell 7C: 진단 전용 — #12 FILT_KEY 확인 + 조기 경보가 '물리 변화'인지 '시간 드리프트'인지 =====
# D-030 선택·결과 변경 없음. LEARNING 미사용. 쓰기: TRAIN_DIR 안 진단 파일 2개만.
from scipy.stats import spearmanr

# --- (1) #12: Next-4 공식 그대로 FILT_KEY 재계산 ---
N4_CUT = {"B_LP_NONE": None, "B_LP8000": 8000.0, "B_LP6000": 6000.0}
def _n4_key(fs_val):
    return hashlib.sha256(json.dumps({"fs": fs_val, "cut": N4_CUT, "ripple": 60.0, "width": 400.0,
        "numtaps": int(numtaps), "beta": float(beta)}, sort_keys=True).encode()).hexdigest()[:12]
k_int, k_flt = _n4_key(int(FS)), _n4_key(float(FS))
FILT_KEY_OK = "7e32cc3e140c" in (k_int, k_flt)
print(f"[#12] Next-4 공식 재계산: fs=int → {k_int} / fs=float → {k_flt} | 7e32cc3e140c 일치: {'✅' if FILT_KEY_OK else '❌'}")
print(f"[#12] LP6000 계수 해시(실제 계수 식별용): {key_try}")

# --- (2) 드리프트 진단 ---
if sel is None: stop("M2ENC_DIAG_NO_SELECTION")
def _sp(a, b):
    r = spearmanr(a, b).correlation
    return float(r) if np.isfinite(r) else np.nan

def _metrics(M, n, t, d0, thr0):
    raw = FN_DET(np.asarray(M, dtype=np.float64)); res = run_detector(M)
    d = np.asarray(raw.get("dists", []), dtype=np.float64); thr = float(raw.get("thr", np.nan))
    nr = int(raw.get("n_reg", REG_MIN)); on = res["onset"]
    ok = len(d) == n and len(d0) == n
    return dict(detected=res["detected"],
                onset_frac=(on / n) if res["detected"] else np.nan,
                rho_time=_sp(d[nr:], t[nr:]) if ok else np.nan,          # 거리 ~ 시간
                rho_m0=_sp(d[nr:], d0[nr:]) if ok else np.nan,           # 거리 ~ M0 물리거리
                m0_ratio_at_onset=float(np.median(d0[on:on + 5]) / thr0)  # 경보 시점 M0 거리 / M0 임계
                    if (res["detected"] and ok) else np.nan)

MODELS = [("M2_" + sel.config, sel["mode"], int(sel.D))]
if sel.config != "KNEE_D16": MODELS.append(("M2_KNEE_D16", "KNEE", 16))
RAND_SEEDS = [0, 1, 2]                      # 학습 안 한 인코더 대조군

rows7c, t0 = [], time.time()
for r in RECORDS[RECORDS.role == "TRAIN"].itertuples():
    v = DATA[r.record_key]; n, t = v["n"], v["t"]
    X = load_record(r); Y = lp6000(X); F = np.stack([feat9(y) for y in Y]); del X, Y
    raw0 = FN_DET(F); d0 = np.asarray(raw0["dists"], dtype=np.float64); thr0 = float(raw0["thr"])
    base = dict(record_key=r.record_key, n=n,
                tail_rms_ratio=float(SCALE.loc[SCALE.record_key == r.record_key, "tail_rms_ratio"].iloc[0]))
    m0 = _metrics(F, n, t, d0, thr0); m0["rho_m0"] = 1.0
    rows7c.append({**base, "model": "M0_PHY", **m0})
    for tag, mode, D in MODELS:
        ck = torch.load(TRAIN_DIR / "ckpt" / f"{mode}_D{D}_f{int(r.fold)}.pt", map_location=DEV)
        net = Net(D).to(DEV); net.load_state_dict(ck["state"])
        rows7c.append({**base, "model": tag, **_metrics(embed(net, v["Z"]), n, t, d0, thr0)})
    for s in RAND_SEEDS:
        torch.manual_seed(1000 + s); net = Net(16).to(DEV)
        rows7c.append({**base, "model": f"RAND_D16_s{s}", **_metrics(embed(net, v["Z"]), n, t, d0, thr0)})
    print(f"  {r.record_key:22s} 완료 ({time.time()-t0:.0f}s)")

DR = pd.DataFrame(rows7c)
DR["group"] = DR.model.str.replace(r"_s\d$", "", regex=True)
DR.to_csv(TRAIN_DIR / "cv_diag_drift.csv", index=False, encoding="utf-8-sig")
AGG_DR = DR.groupby("group").agg(detected=("detected", "sum"), n=("record_key", "count"),
    median_onset_frac=("onset_frac", "median"), median_rho_time=("rho_time", "median"),
    median_rho_m0=("rho_m0", "median"), median_m0_ratio_at_onset=("m0_ratio_at_onset", "median")).reset_index()
(TRAIN_DIR / "CV_DRIFT_DIAG_SUMMARY.json").write_text(json.dumps(dict(
    status="M2ENC_CV_DRIFT_DIAG_COMPLETE", selection_unchanged=True, selected=sel.config,
    filt_key_recomputed={"fs_int": k_int, "fs_float": k_flt, "match_7e32cc3e140c": FILT_KEY_OK},
    lp6000_coef_hash=key_try, m0_feature_fs=FEAT_KW, learning_evaluated=False,
    aggregate=AGG_DR.round(4).to_dict(orient="records")), ensure_ascii=False, indent=2, default=str), encoding="utf-8")

pd.set_option("display.width", 240)
main = DR[~DR.model.str.startswith("RAND")]
print("\n[레코드별: 경보시점 / 경보 때 M0거리비]\n",
      main.pivot_table(index=["record_key", "tail_rms_ratio"], columns="model",
                       values=["onset_frac", "m0_ratio_at_onset"], aggfunc="first").round(3).to_string())
print("\n[레코드별: 거리~시간 상관 / 거리~M0 상관]\n",
      main.pivot_table(index="record_key", columns="model", values=["rho_time", "rho_m0"],
                       aggfunc="first").round(3).to_string())
print("\n[요약] (RAND는 3시드 합산)\n", AGG_DR.round(3).to_string(index=False))
print("\nSTATUS: M2ENC_CV_DRIFT_DIAG_COMPLETE | 선택 변경 없음 | LEARNING 평가: 안 함")


[#12] Next-4 공식 재계산: fs=int → 7e32cc3e140c / fs=float → 54b99597eff0 | 7e32cc3e140c 일치: ✅
[#12] LP6000 계수 해시(실제 계수 식별용): b5543b5a8f89
  FULL_TEST/Bearing1_3   완료 (8s)
  FULL_TEST/Bearing1_4   완료 (11s)
  FULL_TEST/Bearing1_5   완료 (16s)
  FULL_TEST/Bearing1_6   완료 (21s)
  FULL_TEST/Bearing1_7   완료 (25s)
  FULL_TEST/Bearing2_3   완료 (29s)
  FULL_TEST/Bearing2_4   완료 (30s)
  FULL_TEST/Bearing2_5   완료 (35s)
  FULL_TEST/Bearing2_6   완료 (36s)
  FULL_TEST/Bearing2_7   완료 (37s)
  FULL_TEST/Bearing3_3   완료 (38s)

[레코드별: 경보시점 / 경보 때 M0거리비]
                                     m0_ratio_at_onset                        onset_frac                       
model                                          M0_PHY M2_KNEE_D16 M2_LIN_D16     M0_PHY M2_KNEE_D16 M2_LIN_D16
record_key           tail_rms_ratio                                                                           
FULL_TEST/Bearing1_3 4.867040                   1.029       0.747      1.137      0.560       0.470      0.555
FULL_TEST/Bearing1_4 

In [ ]:
# ===== M2_ENC_v1 Cell 8: 최종 모델 재학습 (D-030: 선택 구성으로 FULL_TEST 11개 전체) =====
# LEARNING 미사용. 쓰기: TRAIN_DIR/ckpt/FINAL_*.pt 1개만.
if sel is None or sel.config != "LIN_D16":
    stop("M2ENC_FINAL_SELECTION_UNEXPECTED", sel=None if sel is None else sel.config)
FINAL_MODE, FINAL_D = sel["mode"], int(sel.D)
FINAL_SEED = HP["seed"] + 9999
FINAL_CK = TRAIN_DIR / "ckpt" / f"FINAL_{FINAL_MODE}_D{FINAL_D}.pt"

if FINAL_CK.exists():
    print("  RESUME SKIP", FINAL_CK.name)
else:
    ts = time.time()
    net, hist = train_one(FINAL_MODE, FINAL_D, -1, FINAL_SEED)   # fold_out=-1 → 11개 전부 학습
    tmp = FINAL_CK.with_suffix(".tmp")
    torch.save(dict(state=net.state_dict(), mode=FINAL_MODE, D=FINAL_D, fold="ALL", hist=hist,
                    hp=HP, seed=FINAL_SEED, train_records=sorted(DATA)), tmp)
    os.replace(tmp, FINAL_CK)
    print(f"  {FINAL_CK.name}: 회귀손실 {hist[-1][1]:.4f}, 응집손실 {hist[-1][2]:.4f}, {time.time()-ts:.0f}s")

FINAL_SHA = hashlib.sha256(FINAL_CK.read_bytes()).hexdigest()
_ck = torch.load(FINAL_CK, map_location=DEV)
if _ck["fold"] != "ALL" or len(_ck["train_records"]) != 11: stop("M2ENC_FINAL_CKPT_INVALID")
NET_F = Net(FINAL_D).to(DEV); NET_F.load_state_dict(_ck["state"]); NET_F.eval()

# 학습 데이터 자체 점검(성능 근거 아님 — 임베딩 붕괴만 확인)
_ins = []
for k, v in DATA.items():
    E = embed(NET_F, v["Z"]); _ins.append(dict(record_key=k, detected=run_detector(E)["detected"],
                                               emb_min_std=float(E[:REG_MIN].std(0).min())))
_ins = pd.DataFrame(_ins)
if (_ins.emb_min_std <= 0).any(): stop("M2ENC_FINAL_EMBEDDING_COLLAPSE", bad=_ins[_ins.emb_min_std <= 0].record_key.tolist())
print(f"[FINAL] {FINAL_CK.name}  sha256={FINAL_SHA[:16]}…  seed={FINAL_SEED}")
print(f"[FINAL] 학습데이터 자체 검출 {int(_ins.detected.sum())}/11 (참고용, 성능 근거 아님), 최소 emb_std {_ins.emb_min_std.min():.4f}")
print("Cell 8 완료")


  FINAL_LIN_D16.pt: 회귀손실 0.0179, 응집손실 0.0023, 20s
[FINAL] FINAL_LIN_D16.pt  sha256=508ad40a3fee3ee3…  seed=20271029
[FINAL] 학습데이터 자체 검출 10/11 (참고용, 성능 근거 아님), 최소 emb_std 0.0167
Cell 8 완료


In [ ]:
# ===== M2_ENC_v1 Cell 9: LEARNING 최종 평가 (D-030/D-031, 1회 한정) =====
# 순서: ① M0 재현 게이트(잠금 전) → ② 잠금 생성 → ③ M2 평가 → ④ D-028 판정 + 사전 고정 보고 항목
# 쓰기: TRAIN_DIR/learning_eval_once/ 안만. 동결 파일·원본·데이터셋 쓰기 없음.
from scipy.stats import spearmanr

_req = ["FN_DET", "run_detector", "ns", "lp6000", "load_record", "embed", "NET_F", "FINAL_SHA",
        "FINAL_CK", "RECORDS", "Q_IN", "nb_sha", "FROZEN_NB", "TRAIN_DIR"]
_miss = [n for n in _req if n not in globals()]
if _miss: stop("M2ENC_EVAL_PREREQ_MISSING", missing=_miss, note="Cell 1→2→3→4→5→6→7→8 순서로 실행")

# M0 특징: 동결 extract_features, fs=20480 명시 (Next-4와 동일 경로)
FN_FEAT = ns["extract_features"]
def feat9(x):
    out = FN_FEAT(np.asarray(x, dtype=np.float64), fs=20480)
    return np.asarray(list(out.values()) if isinstance(out, dict) else out, dtype=np.float64).ravel()
def _sp(a, b):
    r = spearmanr(a, b).correlation
    return float(r) if np.isfinite(r) else np.nan

D025_ONSET = {"Bearing1_1": 1462, "Bearing1_2": 826, "Bearing2_1": 874,
              "Bearing2_2": 487, "Bearing3_1": 496, "Bearing3_2": 1584}
D025_FAR   = {"Bearing1_1": 0.0033, "Bearing1_2": 0.0, "Bearing2_1": 0.0044,
              "Bearing2_2": 0.005, "Bearing3_1": 0.0078, "Bearing3_2": 0.0049}   # 소수 4자리 기록값

EVAL_DIR = TRAIN_DIR / "learning_eval_once"; EVAL_DIR.mkdir(exist_ok=True)
LOCK, SUMF = EVAL_DIR / "LEARNING_EVAL.lock", EVAL_DIR / "LEARNING_EVAL_SUMMARY.json"
if SUMF.exists():
    print(SUMF.read_text(encoding="utf-8"))
    stop("M2ENC_LEARNING_ALREADY_EVALUATED", note="1회 한정 — 위 저장 결과를 사용. 재평가 금지")
if LOCK.exists():
    stop("M2ENC_LEARNING_LOCK_WITHOUT_RESULT", lock=LOCK.read_text(encoding="utf-8"),
         note="이전 평가가 중단됨. 잠금 삭제 금지 — 대표 결정 필요")

EV = RECORDS[RECORDS.role == "EVAL"]
if len(EV) != 6 or set(EV.bearing) != set(D025_ONSET): stop("M2ENC_EVAL_SCOPE_MISMATCH", got=EV.bearing.tolist())

# ── ① M0 재현 게이트 ──
M0C, g_rows = {}, []
for r in EV.itertuples():
    X = load_record(r)
    if X.shape != (r.n, NS) or not np.isfinite(X).all(): stop("M2ENC_BAD_SIGNAL", record=r.record_key, shape=X.shape)
    Y = lp6000(X); del X
    F = np.stack([feat9(y) for y in Y])
    raw0 = FN_DET(F); res0 = run_detector(F)
    rms = np.sqrt((Y ** 2).mean(axis=1)); s = float(np.median(rms[:REG_MIN]))       # D-027
    Z = np.clip(Y / s, -Q_IN, Q_IN).astype(np.float32)                                # D-029
    M0C[r.bearing] = dict(n=r.n, Z=Z, s=s, clip_rate=float((np.abs(Y / s) > Q_IN).mean()),
                          d0=np.asarray(raw0["dists"], dtype=np.float64), thr0=float(raw0["thr"]),
                          nreg0=int(raw0["n_reg"]), res0=res0)
    del Y
    lead0 = (r.n - res0["onset"]) * 10 / 3600 if res0["detected"] else np.nan
    g_rows.append(dict(bearing=r.bearing, detected=res0["detected"], onset=res0["onset"], onset_ref=D025_ONSET[r.bearing],
                       reg_far=res0["reg_far"], reg_far_ref=D025_FAR[r.bearing], lead_h=lead0))
G = pd.DataFrame(g_rows)
G["onset_ok"] = G.onset == G.onset_ref
G["far_ok"] = (G.reg_far - G.reg_far_ref).abs() <= 5e-5
m0_lead, m0_far = float(G.lead_h.mean()), float(100 * G.reg_far.mean())
gate_ok = bool(G.detected.all() and G.onset_ok.all() and G.far_ok.all()
               and abs(m0_lead - 0.8356) <= 5e-4 and abs(m0_far - 0.424) <= 5e-4)
G.to_csv(EVAL_DIR / "m0_repro_gate.csv", index=False, encoding="utf-8-sig")
print("[GATE] M0 재현 (D-025 대조)\n", G.round(5).to_string(index=False))
print(f"[GATE] 평균 lead {m0_lead:.4f}h (기준 0.8356) / 평균 reg_FAR {m0_far:.4f}% (기준 0.424) → {'통과 ✅' if gate_ok else '실패 ❌'}")
if not gate_ok:
    stop("M2ENC_M0_REPRO_GATE_FAILED", note="M2 미평가 — 1회 기회 소모 안 됨. 이 출력 그대로 보내주세요")

# ── ② 잠금 ──
LOCK.write_text(json.dumps(dict(created=datetime.datetime.now().isoformat(), final_ckpt=FINAL_CK.name,
    final_sha256=FINAL_SHA, frozen_nb_sha256=nb_sha, decision="D-031 A"), ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\n[LOCK] 생성: {LOCK.name} — 이후 재평가 불가")

# ── ③ M2 평가 + 사전 고정 보고 항목 ──
rows = []
for b, c in M0C.items():
    n = c["n"]; t = np.arange(n) / (n - 1)
    E = embed(NET_F, c["Z"]); raw = FN_DET(E); res = run_detector(E)
    d = np.asarray(raw["dists"], dtype=np.float64); thr = float(raw["thr"]); nr = int(raw["n_reg"])
    on, det = res["onset"], res["detected"]; m0r = c["d0"] / c["thr0"]; on0 = c["res0"]["onset"]
    rows.append(dict(
        bearing=b, n=n, s=c["s"], clip_rate=c["clip_rate"],
        detected=det, onset=on, onset_frac=(on / n) if det else np.nan,
        lead_h=((n - on) * 10 / 3600) if det else np.nan, reg_far_pct=100 * res["reg_far"],
        m0_onset=on0, m0_onset_frac=on0 / n, m2_before_m0=bool(det and on < on0),
        rho_time=_sp(d[nr:], t[nr:]), rho_m0=_sp(d[nr:], c["d0"][nr:]),
        m0_ratio_at_onset=float(np.median(m0r[on:on + 5])) if det else np.nan,
        m0_ratio_reg=float(np.median(m0r[:c["nreg0"]])),
        post_exceed=float((d[on:] > thr).mean()) if det else np.nan,
        emb_min_std=float(E[:REG_MIN].std(0).min())))
R = pd.DataFrame(rows)
R.to_csv(EVAL_DIR / "learning_eval_m2.csv", index=False, encoding="utf-8-sig")

# ── ④ D-028 판정 ──
n_det, m_lead, m_far = int(R.detected.sum()), float(R.lead_h.mean()), float(R.reg_far_pct.mean())
checks = dict(detected_6=n_det >= CRITERIA_D028["detected_min"],
              mean_lead_ok=bool(m_lead >= CRITERIA_D028["mean_lead_h_min"]),
              mean_far_ok=bool(m_far <= CRITERIA_D028["mean_reg_far_pct_max"]))
passed = all(checks.values())
status = "M2ENC_V1_LEARNING_PASS_CONDITIONAL" if passed else "M2ENC_V1_LEARNING_FAIL_MEASURED"
improved = bool(passed and m_lead > BASELINE_D025["mean_lead_h"])
frozen_ok = hashlib.sha256(FROZEN_NB.read_bytes()).hexdigest() == nb_sha

SUMF.write_text(json.dumps(dict(
    status=status, decision="D-031 A (D-030 1회 평가)", criteria_d028=CRITERIA_D028, checks=checks,
    detected=n_det, mean_lead_h=round(m_lead, 4), mean_reg_far_pct=round(m_far, 4),
    improved_vs_d025=improved, baseline_d025=BASELINE_D025,
    m0_repro_gate=dict(passed=True, mean_lead_h=round(m0_lead, 4), mean_reg_far_pct=round(m0_far, 4)),
    report_items_prefixed=["onset_frac", "rho_time", "rho_m0", "m0_ratio_at_onset", "m0_ratio_reg", "m2_before_m0"],
    report_medians=dict(onset_frac=R.onset_frac.median(), rho_time=R.rho_time.median(), rho_m0=R.rho_m0.median(),
                        m0_ratio_at_onset=R.m0_ratio_at_onset.median(), m0_ratio_reg=R.m0_ratio_reg.median()),
    final_ckpt=FINAL_CK.name, final_sha256=FINAL_SHA, train_key=TRAIN_KEY, Q_in=Q_IN,
    filt_key="7e32cc3e140c", lp6000_coef_hash="b5543b5a8f89", m0_feature_fs=20480,
    frozen_notebook_unchanged=frozen_ok, unresolved_open=["#13", "#14"],
    interpretation_limit="FEMTO 조건부 결과. #14(수명 중반 경보 타당성) 미결. 현장 26.667kHz 성능으로 해석 금지",
    per_bearing=R.round(5).to_dict(orient="records")), ensure_ascii=False, indent=2, default=str), encoding="utf-8")

pd.set_option("display.width", 240)
print("\n[LEARNING 베어링별]\n", R[["bearing", "detected", "onset", "m0_onset", "onset_frac", "m0_onset_frac",
      "lead_h", "reg_far_pct", "m2_before_m0"]].round(4).to_string(index=False))
print("\n[사전 고정 보고 항목]\n", R[["bearing", "rho_time", "rho_m0", "m0_ratio_at_onset", "m0_ratio_reg",
      "post_exceed", "emb_min_std", "clip_rate"]].round(4).to_string(index=False))
print(f"\n[D-028] 검출 {n_det}/6 | 평균 lead {m_lead:.4f}h (≥0.752) | 평균 reg_FAR {m_far:.4f}% (≤0.924) → {checks}")
print(f"[D-025 대비] M0 {BASELINE_D025['mean_lead_h']}h vs M2 {m_lead:.4f}h → 개선 {'예' if improved else '아니오'}")
print(f"\nSTATUS: {status} | 동결 노트북 불변: {'✅' if frozen_ok else '❌'} | 저장: {EVAL_DIR}")


[GATE] M0 재현 (D-025 대조)
    bearing  detected  onset  onset_ref  reg_far  reg_far_ref  lead_h  onset_ok  far_ok
Bearing1_1      True   1462       1462  0.00333       0.0033 3.72500      True    True
Bearing1_2      True    826        826  0.00000       0.0000 0.12500      True    True
Bearing2_1      True    874        874  0.00441       0.0044 0.10278      True    True
Bearing2_2      True    487        487  0.00503       0.0050 0.86111      True    True
Bearing3_1      True    496        496  0.00781       0.0078 0.05278      True    True
Bearing3_2      True   1584       1584  0.00489       0.0049 0.14722      True    True
[GATE] 평균 lead 0.8356h (기준 0.8356) / 평균 reg_FAR 0.4244% (기준 0.424) → 통과 ✅

[LOCK] 생성: LEARNING_EVAL.lock — 이후 재평가 불가

[LEARNING 베어링별]
    bearing  detected  onset  m0_onset  onset_frac  m0_onset_frac  lead_h  reg_far_pct  m2_before_m0
Bearing1_1      True   1414      1462      0.5045         0.5216  3.8583       0.3333          True
Bearing1_2      True    571    

In [ ]:
# ============================================================
# Cell 10A — INT8 준비 점검 (읽기 전용: 쓰기·수정·학습 없음)
# ============================================================
import sys, os, glob, inspect, importlib
import numpy as np, torch, torch.nn as nn

print("python", sys.version.split()[0], "| torch", torch.__version__)
for m in ["onnx", "onnxruntime", "onnxscript"]:
    try:
        mod = importlib.import_module(m); print(f"[PKG] {m} {getattr(mod,'__version__','?')}")
    except Exception as e:
        print(f"[PKG] {m} 없음 ({type(e).__name__})")

g = dict(globals())

print("\n[VARS]")
for k in ["TRAIN_DIR","HP","Q_IN","DATASET_ROOT","BRIDGE_ROOT","EXPECTED_SPLIT_SHA",
          "FILT_KEY","SELECTED","SELECTION","FROZEN_SHA_PREFIX"]:
    print(f"  {k} = {repr(g[k])[:200]}" if k in g else f"  {k} : 없음")

print("\n[nn.Module 클래스]")
for k, v in g.items():
    if inspect.isclass(v) and issubclass(v, nn.Module) and v.__module__ == "__main__":
        try: sig = str(inspect.signature(v.__init__))
        except Exception: sig = "?"
        print(f"  {k}{sig}")

print("\n[사용자 정의 함수]")
for k, v in g.items():
    if inspect.isfunction(v) and v.__module__ == "__main__" and not k.startswith("_"):
        try: sig = str(inspect.signature(v))
        except Exception: sig = "?"
        print(f"  {k}{sig}")

print("\n[큰 객체]")
for k, v in g.items():
    if k.startswith("_"): continue
    if isinstance(v, nn.Module):
        print(f"  {k}: {type(v).__name__} params={sum(p.numel() for p in v.parameters())}")
    elif isinstance(v, np.ndarray) and v.size > 1000:
        print(f"  {k}: ndarray {v.shape} {v.dtype}")
    elif isinstance(v, torch.Tensor) and v.numel() > 1000:
        print(f"  {k}: Tensor {tuple(v.shape)} {v.dtype}")
    elif isinstance(v, dict) and k.isupper():
        print(f"  {k}: dict({len(v)}) keys={list(v)[:12]}")
    elif isinstance(v, (list, tuple)) and k.isupper() and 0 < len(v) <= 40:
        print(f"  {k}: {type(v).__name__}({len(v)}) {repr(v)[:150]}")

if "TRAIN_DIR" in g:
    ck = os.path.join(str(TRAIN_DIR), "ckpt", "FINAL_LIN_D16.pt")
    print("\n[CKPT]", ck, "존재" if os.path.exists(ck) else "없음")
    if os.path.exists(ck):
        obj = torch.load(ck, map_location="cpu", weights_only=False)
        print("  type:", type(obj).__name__)
        if isinstance(obj, dict):
            print("  keys:", list(obj)[:20])
            sd = obj.get("model", obj.get("state_dict", obj))
            if isinstance(sd, dict):
                for kk, vv in list(sd.items())[:40]:
                    if hasattr(vv, "shape"): print(f"    {kk} {tuple(vv.shape)}")
    locks = glob.glob(os.path.join(str(TRAIN_DIR), "**", "LEARNING_EVAL.lock"), recursive=True)
    print("\n[LOCK]", locks if locks else "못 찾음")
    ev = os.path.join(str(TRAIN_DIR), "learning_eval_once")
    if os.path.isdir(ev): print("[EVAL_DIR 파일]", sorted(os.listdir(ev))[:30])

print("\nCell 10A 완료 — 출력 전체를 보내주세요 (아무것도 저장·수정하지 않았습니다)")


python 3.13.16 | torch 2.11.0+cu130
[PKG] onnx 없음 (ModuleNotFoundError)
[PKG] onnxruntime 없음 (ModuleNotFoundError)
[PKG] onnxscript 없음 (ModuleNotFoundError)

[VARS]
  TRAIN_DIR = PosixPath('/content/drive/MyDrive/Colab Notebooks/field_iis3dwb/bridge_outputs/m2_enc_v1_train_42f861b9506e')
  HP = {'epochs': 30, 'batch': 256, 'lr': 0.001, 'wd': 0.0001, 'lam_compact': 0.1, 'normal_frac': 0.3, 'knee': 0.7, 'seed': 20261030, 'aug': 'roll+signflip', 'arch': 'v1a'}
  Q_IN = 8.0
  DATASET_ROOT : 없음
  BRIDGE_ROOT : 없음
  EXPECTED_SPLIT_SHA = '41a0e1f243eb1fa98b4d8dfc894b281ab843e6faf7f9d5902b7f0a97d790c10e'
  FILT_KEY : 없음
  SELECTED : 없음
  SELECTION : 없음
  FROZEN_SHA_PREFIX = '921f1bbea7b460d748c6dc3f74c04981f3bf4de5ff76223b'

[nn.Module 클래스]
  Enc(self, D)
  Net(self, D)

[사용자 정의 함수]
  stop(code, **kw)
  pick(df, cands, what, required=True)
  norm_split(v)
  norm_bearing(v)
  find_npy(split, bearing, idx_file)
  lp6000(X)
  load_record(r, tries=5)
  run_detector(E)
  blk(ci, co, k, s, p)
  targ

In [ ]:
# ===== M2_ENC_v1 Cell 10 (r1): INT8 변환 — ONNX 내보내기 + 게이트0 + QDQ 정적 양자화 (D-032) =====
# LEARNING 읽지 않음. 쓰기: TRAIN_DIR/int8_v1/ 안만. 동결 파일·잠금·체크포인트 수정 없음.
import subprocess, sys, json, hashlib, time, collections
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "onnx", "onnxruntime", "onnxscript"], check=True)
import numpy as np, torch, onnx, onnxruntime as ort
from onnx import numpy_helper
from onnxruntime.quantization import (quantize_static, QuantFormat, QuantType,
                                      CalibrationDataReader, CalibrationMethod)
from onnxruntime.quantization.shape_inference import quant_pre_process

_req = ["DATA", "RECORDS", "Net", "TRAIN_DIR", "HP", "Q_IN", "NS", "stop", "Path"]
_miss = [n for n in _req if n not in globals()]
if _miss: raise RuntimeError(f"선행 셀 변수 없음 {_miss} → Cell 1~6 먼저 실행")

INT8_DIR = TRAIN_DIR / "int8_v1"; INT8_DIR.mkdir(exist_ok=True)
P32, PPRE, P8 = INT8_DIR / "enc_fp32.onnx", INT8_DIR / "enc_fp32_pre.onnx", INT8_DIR / "enc_int8_qdq.onnx"
def _sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
print(f"[ENV] torch {torch.__version__} | onnx {onnx.__version__} | onnxruntime {ort.__version__}")

# ── 최종 모델 로드 (Cell 8 산출물, 재학습 없음) ──
FINAL_CK = TRAIN_DIR / "ckpt" / "FINAL_LIN_D16.pt"
FINAL_SHA = _sha(FINAL_CK)
if not FINAL_SHA.startswith("508ad40a3fee3ee3"): stop("M2ENC_INT8_FINAL_CKPT_CHANGED", sha=FINAL_SHA)
_ck = torch.load(FINAL_CK, map_location="cpu", weights_only=False)
if _ck["mode"] != "LIN" or int(_ck["D"]) != 16 or _ck["fold"] != "ALL":
    stop("M2ENC_INT8_FINAL_CKPT_INVALID", mode=_ck["mode"], D=_ck["D"], fold=_ck["fold"])
D_EMB = 16
NET_CPU = Net(D_EMB); NET_CPU.load_state_dict(_ck["state"]); NET_CPU.eval()
ENC = NET_CPU.enc.eval()
print(f"[FINAL] {FINAL_CK.name} sha256={FINAL_SHA[:16]}… 인코더 파라미터 {sum(p.numel() for p in ENC.parameters())}개")

@torch.no_grad()
def embed_cpu(Z, bs=512):
    return np.concatenate([ENC(torch.from_numpy(np.ascontiguousarray(Z[i:i + bs])).unsqueeze(1)).double().numpy()
                           for i in range(0, len(Z), bs)])

# ── ONNX 내보내기 (배치 1 고정 = NPU 실행 형태) ──
x0 = torch.zeros(1, 1, NS)
try:
    torch.onnx.export(ENC, (x0,), str(P32), input_names=["input"], output_names=["embedding"],
                      opset_version=17, do_constant_folding=True, dynamo=False)
    EXPORT_MODE = "torchscript_opset17"
except Exception as e:
    print("[EXPORT] torchscript 경로 실패 → dynamo 경로:", repr(e)[:200])
    torch.onnx.export(ENC, (x0,), str(P32), input_names=["input"], output_names=["embedding"],
                      opset_version=18, dynamo=True)
    EXPORT_MODE = "dynamo_opset18"
onnx.checker.check_model(onnx.load(str(P32)))
OPS32 = collections.Counter(n.op_type for n in onnx.load(str(P32)).graph.node)
print(f"[EXPORT] {EXPORT_MODE} | FP32 연산: {dict(OPS32)}")

def mk_sess(p): return ort.InferenceSession(str(p), providers=["CPUExecutionProvider"])
def ort_embed(sess, Z):
    nm = sess.get_inputs()[0].name; out = np.empty((len(Z), D_EMB), np.float64)
    for i in range(len(Z)):
        out[i] = sess.run(None, {nm: np.ascontiguousarray(Z[i:i + 1, None, :])})[0].reshape(-1)
    return out

# ── 게이트0: FP32 ONNX vs PyTorch(CPU), FULL_TEST 11개 전 윈도 ──
s32 = mk_sess(P32); IN_NAME = s32.get_inputs()[0].name
G0 = {k: float(np.abs(ort_embed(s32, DATA[k]["Z"]) - embed_cpu(DATA[k]["Z"])).max()) for k in sorted(DATA)}
G0_MAX = max(G0.values())
print(f"[GATE0] FP32 ONNX vs PyTorch 최대 절대오차 {G0_MAX:.2e} (기준 ≤ 1e-4) → {'통과 ✅' if G0_MAX <= 1e-4 else '실패 ❌'}")
if G0_MAX > 1e-4: stop("M2ENC_INT8_GATE0_FAILED", per_record=G0, note="양자화 안 함")

# ── 캘리브레이션 데이터: FULL_TEST 11 × 200윈도 (seed 고정, LEARNING 미사용) ──
rng = np.random.default_rng(HP["seed"])
CAL_IDX = {k: np.sort(rng.choice(DATA[k]["n"], size=200, replace=False)) for k in sorted(DATA)}
CAL = np.concatenate([DATA[k]["Z"][CAL_IDX[k]] for k in sorted(DATA)]).astype(np.float32)
if CAL.shape != (2200, NS): stop("M2ENC_INT8_CAL_SIZE", shape=CAL.shape)

# ── 전처리(BN 융합·형상 추론) + 재확인 ──
try:
    quant_pre_process(str(P32), str(PPRE), skip_symbolic_shape=True); PQ_IN, PRE_OK = PPRE, True
except Exception as e:
    print("[PRE] 전처리 실패 → 원본 FP32로 양자화:", repr(e)[:200]); PQ_IN, PRE_OK = P32, False
PRE_ERR = float(np.abs(ort_embed(mk_sess(PQ_IN), CAL) - embed_cpu(CAL)).max())
print(f"[PRE] 전처리 {'적용' if PRE_OK else '미적용'} | 전처리본 vs PyTorch 최대오차 {PRE_ERR:.2e}")
if PRE_ERR > 1e-4: stop("M2ENC_INT8_PRE_GATE_FAILED", err=PRE_ERR)

class CalReader(CalibrationDataReader):
    def __init__(self, X, name): self.X, self.name, self.i = X, name, 0
    def get_next(self):
        if self.i >= len(self.X): return None
        x = np.ascontiguousarray(self.X[self.i:self.i + 1, None, :]); self.i += 1
        return {self.name: x}
    def rewind(self): self.i = 0

# ── QDQ 정적 양자화: 가중치 INT8 채널별 대칭 / 활성 INT8 텐서 단위 / 입력 scale 8/127·zp 0 고정(D-029) ──
# [r2] onnxruntime은 override 값으로 numpy 스칼라(np.int8 등)를 받지 않음 → 0차원 numpy 배열로 전달
S_IN = np.float32(Q_IN / 127.0)
IN_OVERRIDE = {"scale": np.array(S_IN, dtype=np.float32),
               "zero_point": np.array(0, dtype=np.int8)}
quantize_static(str(PQ_IN), str(P8), CalReader(CAL, IN_NAME),
                quant_format=QuantFormat.QDQ, per_channel=True,
                weight_type=QuantType.QInt8, activation_type=QuantType.QInt8,
                calibrate_method=CalibrationMethod.MinMax,
                extra_options=dict(WeightSymmetric=True, ActivationSymmetric=False,
                                   TensorQuantOverrides={IN_NAME: [IN_OVERRIDE]}))

# ── 결과 구조 확인 ──
m8 = onnx.load(str(P8)); onnx.checker.check_model(m8)
inits = {t.name: numpy_helper.to_array(t) for t in m8.graph.initializer}
qin = [n for n in m8.graph.node if n.op_type == "QuantizeLinear" and n.input[0] == IN_NAME]
if len(qin) != 1 or qin[0].input[1] not in inits or qin[0].input[2] not in inits:
    stop("M2ENC_INT8_INPUT_Q_NOT_FOUND", n=len(qin))
s_in, zp_in = float(inits[qin[0].input[1]]), inits[qin[0].input[2]]
IN_Q_OK = abs(s_in - float(S_IN)) <= 1e-7 and int(zp_in) == 0 and zp_in.dtype == np.int8
OPS8 = collections.Counter(n.op_type for n in m8.graph.node)
n_perch = sum(1 for n in m8.graph.node if n.op_type == "DequantizeLinear"
              and n.input[1] in inits and inits[n.input[1]].size > 1)
print(f"[INT8] 입력 scale={s_in:.8f} (목표 {float(S_IN):.8f}) zp={int(zp_in)} {zp_in.dtype} → {'✅' if IN_Q_OK else '❌'}")
print(f"[INT8] 채널별 가중치 텐서 {n_perch}개 | BN 잔존 {OPS8.get('BatchNormalization', 0)}개 | 연산: {dict(OPS8)}")
if not IN_Q_OK: stop("M2ENC_INT8_INPUT_SCALE_NOT_FIXED", scale=s_in, zp=str(zp_in))

summary10 = dict(status="M2ENC_INT8_BUILT_AWAITING_EQUIV", decision="D-032 A",
    versions=dict(torch=torch.__version__, onnx=onnx.__version__, onnxruntime=ort.__version__, python=sys.version.split()[0]),
    final_ckpt=FINAL_CK.name, final_sha256=FINAL_SHA, export_mode=EXPORT_MODE,
    gate0_max_abs_err=G0_MAX, gate0_per_record=G0, pre_process=PRE_OK, pre_max_abs_err=PRE_ERR,
    calibration=dict(source="FULL_TEST 11 x 200", seed=HP["seed"], method="MinMax", n=int(len(CAL))),
    input_q=dict(scale=s_in, zero_point=int(zp_in)), per_channel_weight_tensors=n_perch,
    ops_fp32=dict(OPS32), ops_int8=dict(OPS8),
    sha256=dict(fp32=_sha(P32), int8=_sha(P8)), learning_used=False)
(INT8_DIR / "INT8_BUILD_SUMMARY.json").write_text(json.dumps(summary10, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print(f"\n[SHA256] fp32={summary10['sha256']['fp32'][:16]}… int8={summary10['sha256']['int8'][:16]}…")
print("STATUS: M2ENC_INT8_BUILT_AWAITING_EQUIV | LEARNING 사용: 없음 | Cell 10 완료")


[ENV] torch 2.11.0+cu130 | onnx 1.23.2 | onnxruntime 1.31.1
[FINAL] FINAL_LIN_D16.pt sha256=508ad40a3fee3ee3… 인코더 파라미터 8056개


/tmp/ipykernel_2377/3899783939.py:40: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(ENC, (x0,), str(P32), input_names=["input"], output_names=["embedding"],


[EXPORT] torchscript_opset17 | FP32 연산: {'Conv': 5, 'Relu': 5, 'GlobalAveragePool': 1, 'Flatten': 1, 'Gemm': 1}
[GATE0] FP32 ONNX vs PyTorch 최대 절대오차 7.63e-06 (기준 ≤ 1e-4) → 통과 ✅
[PRE] 전처리 적용 | 전처리본 vs PyTorch 최대오차 7.63e-06
[INT8] 입력 scale=0.06299213 (목표 0.06299213) zp=0 int8 → ✅
[INT8] 채널별 가중치 텐서 12개 | BN 잔존 0개 | 연산: {'DequantizeLinear': 21, 'QuantizeLinear': 9, 'Conv': 5, 'GlobalAveragePool': 1, 'Flatten': 1, 'Gemm': 1}

[SHA256] fp32=294c1fab84de0bb8… int8=ee91f648cc02e369…
STATUS: M2ENC_INT8_BUILT_AWAITING_EQUIV | LEARNING 사용: 없음 | Cell 10 완료


In [ ]:
# ===== M2_ENC_v1 Cell 10B (r1): INT8 실패 원인 진단 + 캘리브레이션 변형 비교 (D-032) =====
# FULL_TEST만 사용, LEARNING 미사용 → 시도 소모 없음. 쓰기: TRAIN_DIR/int8_v1/variants_10b/ 안만.
# 사전 고정 선택 규칙: ①11/11 onset_ok ②전부 INT8 우선(Gemm FP는 대표 승인 필요) ③cos_min 큰 순 ④최대 onset 차 작은 순
import json, collections, numpy as np, pandas as pd, onnx
from onnx import numpy_helper
from onnxruntime.quantization import quantize_static, QuantFormat, QuantType, CalibrationMethod

_req = ["CalReader", "CAL", "IN_NAME", "PQ_IN", "IN_OVERRIDE", "mk_sess", "ort_embed", "run_detector",
        "FN_DET", "DATA", "INT8_DIR", "P32", "P8", "_sha", "D_EMB", "REG_MIN", "stop"]
_miss = [n for n in _req if n not in globals()]
if _miss: raise RuntimeError(f"선행 변수 없음 {_miss} → Cell 1~6, Cell 10 먼저 실행")

VAR_DIR = INT8_DIR / "variants_10b"; VAR_DIR.mkdir(exist_ok=True)
KEYS = sorted(DATA)

# ── FP32 ONNX 기준값 캐시 ──
S32 = mk_sess(P32)
E32 = {k: ort_embed(S32, DATA[k]["Z"]) for k in KEYS}
RAW32 = {k: FN_DET(E32[k]) for k in KEYS}
R32 = {k: run_detector(E32[k]) for k in KEYS}
REG_STD_MIN = min(float(E32[k][:REG_MIN].std(0).min()) for k in KEYS)
print(f"[FP32] 등록구간 임베딩 최소 표준편차 {REG_STD_MIN:.4f} | 전 구간 임베딩 |최대| "
      f"{max(float(np.abs(E32[k]).max()) for k in KEYS):.3f}")

def act_scales(p):
    m = onnx.load(str(p)); ini = {t.name: numpy_helper.to_array(t) for t in m.graph.initializer}
    return [(n.input[0], float(np.asarray(ini[n.input[1]]).ravel()[0]))
            for n in m.graph.node if n.op_type == "QuantizeLinear" and n.input[1] in ini]

def emb_step(p):
    m = onnx.load(str(p)); g = [n for n in m.graph.node if n.op_type == "Gemm"]
    if not g: return None
    return next((s for name, s in act_scales(p) if name == g[0].output[0]), None)  # None = Gemm 출력 FP

GEMM_NODES = [n.name for n in onnx.load(str(PQ_IN)).graph.node if n.op_type == "Gemm"]
if len(GEMM_NODES) != 1 or not GEMM_NODES[0]: stop("M2ENC_INT8_GEMM_NODE_NAME", nodes=GEMM_NODES)

VARIANTS = [
    ("V1_MINMAX",          CalibrationMethod.MinMax,     {},                          []),
    ("V2_PCT99999",        CalibrationMethod.Percentile, {"CalibPercentile": 99.999}, []),
    ("V3_PCT9999",         CalibrationMethod.Percentile, {"CalibPercentile": 99.99},  []),
    ("V4_ENTROPY",         CalibrationMethod.Entropy,    {},                          []),
    ("V5_MINMAX_GEMMFP",   CalibrationMethod.MinMax,     {},                          GEMM_NODES),
    ("V6_PCT99999_GEMMFP", CalibrationMethod.Percentile, {"CalibPercentile": 99.999}, GEMM_NODES),
]

def eval_variant(p):
    s8, rows = mk_sess(p), []
    for k in KEYS:
        E8 = ort_embed(s8, DATA[k]["Z"]); raw8 = FN_DET(E8); r8 = run_detector(E8); r32 = R32[k]
        cos = (E32[k] * E8).sum(1) / (np.linalg.norm(E32[k], axis=1) * np.linalg.norm(E8, axis=1) + 1e-12)
        same = r32["detected"] == r8["detected"]
        d = abs(r32["onset"] - r8["onset"]) if (r32["detected"] and r8["detected"]) else None
        d32 = np.asarray(RAW32[k]["dists"], float); d8 = np.asarray(raw8["dists"], float)
        nt = max(1, int(0.1 * len(d32)))
        rows.append(dict(record_key=k, onset_fp32=r32["onset"], onset_int8=r8["onset"], onset_diff=d,
                         onset_ok=bool(same and (d is None or d <= 5)),
                         far_int8_pct=100 * r8["reg_far"], cos_median=float(np.median(cos)), cos_min=float(cos.min()),
                         reg_std_ratio=float(E8[:REG_MIN].std(0).min() / max(E32[k][:REG_MIN].std(0).min(), 1e-12)),
                         thr_ratio=float(raw8["thr"]) / float(RAW32[k]["thr"]),
                         tail_dist_ratio=float(np.median(d8[-nt:]) / max(np.median(d32[-nt:]), 1e-12))))
    return pd.DataFrame(rows)

summ, per = [], {}
for name, meth, extra, excl in VARIANTS:
    p = VAR_DIR / f"enc_int8_{name}.onnx"
    try:
        quantize_static(str(PQ_IN), str(p), CalReader(CAL, IN_NAME),
                        quant_format=QuantFormat.QDQ, per_channel=True,
                        weight_type=QuantType.QInt8, activation_type=QuantType.QInt8,
                        calibrate_method=meth, nodes_to_exclude=list(excl),
                        extra_options=dict(WeightSymmetric=True, ActivationSymmetric=False,
                                           TensorQuantOverrides={IN_NAME: [dict(IN_OVERRIDE)]}, **extra))
        sc = dict(act_scales(p)); s_in = sc.get(IN_NAME, np.nan)
        if abs(s_in - float(IN_OVERRIDE["scale"])) > 1e-7: raise RuntimeError(f"입력 scale 고정 실패 {s_in}")
        df = eval_variant(p); per[name] = df
        df.to_csv(VAR_DIR / f"equiv_fulltest_{name}.csv", index=False, encoding="utf-8-sig")
        es = emb_step(p)
        summ.append(dict(variant=name, gemm_fp=bool(excl), n_ok=int(df.onset_ok.sum()),
                         max_onset_diff=float(pd.to_numeric(df.onset_diff, errors="coerce").max()),
                         cos_median=float(df.cos_median.median()), cos_min=float(df.cos_min.min()),
                         reg_std_ratio_med=float(df.reg_std_ratio.median()),
                         tail_dist_ratio_med=float(df.tail_dist_ratio.median()),
                         emb_step=es, emb_step_over_regstd=(es / REG_STD_MIN) if es else None,
                         sha256=_sha(p)[:16], path=str(p), error=None))
        print(f"  {name:20s} onset_ok {summ[-1]['n_ok']}/11  cos_min {summ[-1]['cos_min']:.3f}")
    except Exception as e:
        summ.append(dict(variant=name, gemm_fp=bool(excl), n_ok=-1, error=repr(e)[:200], path=str(p)))
        print(f"  {name:20s} 실패: {repr(e)[:150]}")

SUMM = pd.DataFrame(summ)
SUMM.to_csv(VAR_DIR / "variants_summary.csv", index=False, encoding="utf-8-sig")
pd.set_option("display.width", 240)
print("\n[V1 활성 scale (텐서, scale)]"); [print(f"  {n:40s} {s:.6f}") for n, s in act_scales(VAR_DIR / "enc_int8_V1_MINMAX.onnx")]
print("\n[변형 요약]\n", SUMM.drop(columns=["path"]).round(4).to_string(index=False))
for name in ["V1_MINMAX"] + [v for v in per if v != "V1_MINMAX"][:0]:
    print(f"\n[{name} 레코드별]\n", per[name].round(4).to_string(index=False))

cand = SUMM[SUMM.n_ok == 11].sort_values(["gemm_fp", "cos_min", "max_onset_diff"], ascending=[True, False, True])
if cand.empty:
    SEL, STATUS = None, "M2ENC_INT8_CAL_VARIANTS_NONE_PASS_QAT_NEXT"
elif not cand.iloc[0].gemm_fp:
    SEL, STATUS = cand.iloc[0], "M2ENC_INT8_CAL_VARIANT_SELECTED"
    P8 = Path(SEL.path)   # Cell 11에서 이 모델을 사용
else:
    SEL, STATUS = cand.iloc[0], "M2ENC_INT8_ONLY_GEMMFP_PASS_AWAITING_DECISION"
(VAR_DIR / "CAL_VARIANTS_SUMMARY.json").write_text(json.dumps(dict(
    status=STATUS, learning_used=False, attempt_consumed=False, reg_std_min_fp32=REG_STD_MIN,
    selection_rule="11/11 onset_ok → full INT8 우선 → cos_min desc → max_onset_diff asc",
    selected=None if SEL is None else SEL.variant, variants=SUMM.round(5).to_dict(orient="records")),
    ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print(f"\nSTATUS: {STATUS} | 선택: {None if SEL is None else SEL.variant} | "
      f"P8 → {P8.name if STATUS == 'M2ENC_INT8_CAL_VARIANT_SELECTED' else '변경 없음'} | LEARNING 사용: 없음")



[FP32] 등록구간 임베딩 최소 표준편차 0.0167 | 전 구간 임베딩 |최대| 38.366
  V1_MINMAX            onset_ok 1/11  cos_min 0.844
Finding optimal threshold for each tensor using 'percentile' algorithm ...
Number of tensors : 14
Number of histogram bins : 2048
Percentile : (0.0010000000000047748,99.999)
  V2_PCT99999          onset_ok 1/11  cos_min 0.833
Finding optimal threshold for each tensor using 'percentile' algorithm ...
Number of tensors : 14
Number of histogram bins : 2048
Percentile : (0.010000000000005116,99.99)
  V3_PCT9999           onset_ok 2/11  cos_min 0.861
Finding optimal threshold for each tensor using 'entropy' algorithm ...
Number of tensors : 14
Number of histogram bins : 128 (The number may increase depends on the data it collects)
Number of quantized bins : 128
  V4_ENTROPY           onset_ok 1/11  cos_min 0.844
  V5_MINMAX_GEMMFP     onset_ok 4/11  cos_min 0.941
Finding optimal threshold for each tensor using 'percentile' algorithm ...
Number of tensors : 14
Number of histogram bins : 

In [ ]:
# ===== M2_ENC_v1 Cell 10C (r1): 비트 수 진단 — PyTorch 모의 양자화 (D-032 진단, FULL_TEST만) =====
# LEARNING 미사용, 시도 소모 없음. 쓰기: TRAIN_DIR/int8_v1/bits_sweep_10c/ 안만. 모델·ONNX 변경 없음.
import json, numpy as np, pandas as pd, torch, torch.nn.functional as Fnn
_req = ["ENC", "DATA", "CAL", "R32", "E32", "run_detector", "INT8_DIR", "Q_IN", "REG_MIN", "KEYS", "stop"]
_miss = [n for n in _req if n not in globals()]
if _miss: raise RuntimeError(f"선행 변수 없음 {_miss} → Cell 10, 10B 먼저 실행")
OUT10C = INT8_DIR / "bits_sweep_10c"; OUT10C.mkdir(exist_ok=True)

# ── BN을 Conv에 융합 (ONNX와 같은 구조) ──
with torch.no_grad():
    LAYERS = []
    for i in range(5):
        conv, bn = ENC.f[i][0], ENC.f[i][1]
        g = bn.weight / torch.sqrt(bn.running_var + bn.eps)
        LAYERS.append(((conv.weight * g[:, None, None]).double(), (bn.bias - bn.running_mean * g).double(),
                       conv.stride[0], conv.padding[0]))
    WE, BE = ENC.emb.weight.double(), ENC.emb.bias.double()
def qw(w):  # INT8 채널별 대칭
    s = (w.abs().amax(dim=tuple(range(1, w.dim())), keepdim=True) / 127).clamp_min(1e-12)
    return torch.round(w / s).clamp(-127, 127) * s
WQ, WEQ = [qw(L[0]) for L in LAYERS], qw(WE)
PTS = ["L0", "L1", "L2", "L3", "L4", "POOL", "EMB"]
RNG = {}

def fq(t, name, cfg):
    b = cfg.get(name)
    if not b: return t
    lo, hi = min(RNG[name][0], 0.0), max(RNG[name][1], 0.0)
    s = (hi - lo) / (2 ** b - 1); zp = round(-lo / s)
    return (torch.clamp(torch.round(t / s) + zp, 0, 2 ** b - 1) - zp) * s

@torch.no_grad()
def sim(X, cfg, wq, qin, collect=None):
    x = torch.from_numpy(np.ascontiguousarray(X)).double().unsqueeze(1)
    if qin: s = Q_IN / 127; x = torch.round(x / s).clamp(-128, 127) * s
    def tap(t, n):
        if collect is not None:
            lo, hi = collect.get(n, (np.inf, -np.inf)); collect[n] = (min(lo, float(t.min())), max(hi, float(t.max())))
        return fq(t, n, cfg)
    for i, (w, b, st, pd_) in enumerate(LAYERS):
        x = tap(Fnn.relu(Fnn.conv1d(x, WQ[i] if wq else w, b, stride=st, padding=pd_)), f"L{i}")
    x = tap(x.mean(-1), "POOL")
    return tap(x @ (WEQ if wq else WE).T + BE, "EMB").numpy()

def emb_all(Z, cfg, wq, qin, bs=1024):
    return np.concatenate([sim(Z[i:i + bs], cfg, wq, qin) for i in range(0, len(Z), bs)])

# ── 게이트: 양자화 없는 모의 = FP32 ONNX ──
g_err = max(float(np.abs(emb_all(DATA[k]["Z"], {}, False, False) - E32[k]).max()) for k in KEYS)
print(f"[GATE] 모의(FP) vs FP32 ONNX 최대오차 {g_err:.2e}")
if g_err > 1e-4: stop("M2ENC_10C_SIM_MISMATCH", err=g_err)

# ── 활성 범위: 캘리브레이션 2,200윈도(FULL_TEST)로 MinMax ──
for i in range(0, len(CAL), 512): sim(CAL[i:i + 512], {}, True, True, collect=RNG)
print("[RANGE]", {k: (round(v[0], 3), round(v[1], 3)) for k, v in RNG.items()})

A8 = {p: 8 for p in PTS}
CFGS = [("S1_W8A8", A8, True), ("S2_A8_EMB16", {**A8, "EMB": 16}, True),
        ("S3_A8_POOL16_EMB16", {**A8, "POOL": 16, "EMB": 16}, True),
        ("S4_W8A10", {p: 10 for p in PTS}, True), ("S5_W8A12", {p: 12 for p in PTS}, True),
        ("S6_W8A16", {p: 16 for p in PTS}, True), ("S7_W8_ONLY", {}, True)]

summ, per = [], []
for name, cfg, wq in CFGS:
    rows = []
    for k in KEYS:
        E = emb_all(DATA[k]["Z"], cfg, wq, True); r8, r32 = run_detector(E), R32[k]
        same = r8["detected"] == r32["detected"]
        d = abs(r8["onset"] - r32["onset"]) if (r8["detected"] and r32["detected"]) else None
        cos = (E * E32[k]).sum(1) / (np.linalg.norm(E, axis=1) * np.linalg.norm(E32[k], axis=1) + 1e-12)
        rows.append(dict(cfg=name, record_key=k, onset_fp32=r32["onset"], onset_q=r8["onset"], onset_diff=d,
                         onset_ok=bool(same and (d is None or d <= 5)), cos_min=float(cos.min()),
                         reg_std_ratio=float(E[:REG_MIN].std(0).min() / max(E32[k][:REG_MIN].std(0).min(), 1e-12))))
    df = pd.DataFrame(rows); per.append(df)
    summ.append(dict(cfg=name, n_ok=int(df.onset_ok.sum()),
                     max_onset_diff=float(pd.to_numeric(df.onset_diff, errors="coerce").max()),
                     cos_min=float(df.cos_min.min()), reg_std_ratio_min=float(df.reg_std_ratio.min())))
    print(f"  {name:22s} onset_ok {summ[-1]['n_ok']}/11  max_diff {summ[-1]['max_onset_diff']:.0f}  cos_min {summ[-1]['cos_min']:.3f}")

S = pd.DataFrame(summ); PER = pd.concat(per)
S.to_csv(OUT10C / "bits_sweep_summary.csv", index=False, encoding="utf-8-sig")
PER.to_csv(OUT10C / "bits_sweep_per_record.csv", index=False, encoding="utf-8-sig")
ok = dict(zip(S.cfg, S.n_ok == 11))
if not ok["S7_W8_ONLY"]:           VERDICT = "WEIGHTS_ALSO_LIMIT_QAT_REQUIRED"
elif ok["S2_A8_EMB16"] or ok["S3_A8_POOL16_EMB16"]: VERDICT = "MIXED_PRECISION_TAIL_SUFFICIENT"
elif ok["S4_W8A10"] or ok["S5_W8A12"]:              VERDICT = "NEEDS_10_12BIT_QAT_GAP_LARGE"
elif ok["S6_W8A16"]:               VERDICT = "NEEDS_16BIT_QAT_UNLIKELY"
else:                              VERDICT = "DETECTOR_HYPERSENSITIVE_DECISION_NEEDED"
(OUT10C / "BITS_SWEEP_SUMMARY.json").write_text(json.dumps(dict(status="M2ENC_INT8_BITS_SWEEP_COMPLETE",
    verdict=VERDICT, gate_err=g_err, ranges=RNG, summary=S.round(5).to_dict(orient="records"),
    learning_used=False, attempt_consumed=False), ensure_ascii=False, indent=2, default=str), encoding="utf-8")
pd.set_option("display.width", 220)
print("\n[요약]\n", S.round(4).to_string(index=False))
print(f"\nSTATUS: M2ENC_INT8_BITS_SWEEP_COMPLETE | 판정: {VERDICT} | LEARNING 사용: 없음")


[GATE] 모의(FP) vs FP32 ONNX 최대오차 8.46e-06
[RANGE] {'L0': (0.0, 23.835), 'L1': (0.0, 17.391), 'L2': (0.0, 15.927), 'L3': (0.0, 13.618), 'L4': (0.0, 17.468), 'POOL': (0.0, 12.899), 'EMB': (-38.356, 17.973)}
  S1_W8A8                onset_ok 1/11  max_diff 1347  cos_min 0.871
  S2_A8_EMB16            onset_ok 7/11  max_diff 385  cos_min 0.991
  S3_A8_POOL16_EMB16     onset_ok 7/11  max_diff 40  cos_min 0.997
  S4_W8A10               onset_ok 4/11  max_diff 473  cos_min 0.990
  S5_W8A12               onset_ok 6/11  max_diff 86  cos_min 0.999
  S6_W8A16               onset_ok 11/11  max_diff 0  cos_min 0.999
  S7_W8_ONLY             onset_ok 11/11  max_diff 1  cos_min 0.999

[요약]
                cfg  n_ok  max_onset_diff  cos_min  reg_std_ratio_min
           S1_W8A8     1          1347.0   0.8712             0.0000
       S2_A8_EMB16     7           385.0   0.9905             0.9795
S3_A8_POOL16_EMB16     7            40.0   0.9966             0.9892
          S4_W8A10     4           473.0

In [ ]:
# ================================================================
# Cell 10D — D-033: w8a16 모델 생성 (r1)
#  가중치 INT8 채널별 대칭 / 활성 INT16 대칭(zp=0) / 활성 scale = 2의 거듭제곱(올림)
#  입력 scale = 2^-12 고정 (Q_IN=8 → ±8, D-029 유지)
#  LEARNING 읽지 않음 | 쓰기: TRAIN_DIR/int8_v1/w8a16_10d/ 만 | 판정 없음
# ================================================================
import json, math, hashlib, time
import numpy as np
from pathlib import Path
import onnx
from onnx import numpy_helper
import onnxruntime as ort
from onnxruntime.quantization import (quantize_static, QuantType, QuantFormat,
                                      CalibrationMethod, CalibrationDataReader)
from onnxruntime.quantization.shape_inference import quant_pre_process

def _stop10d(code, msg):
    raise RuntimeError(f"{code}: {msg}")

for _n in ["TRAIN_DIR", "Q_IN", "P32"]:
    if _n not in globals():
        _stop10d("M2ENC_W8A16_PREREQ_MISSING", f"'{_n}' 없음 → Cell 1~6, 10 실행 후 10D")
if abs(float(Q_IN) - 8.0) > 1e-12:
    _stop10d("M2ENC_W8A16_QIN_MISMATCH", f"Q_IN={Q_IN} (D-029는 8.0)")

def _sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

P32 = Path(P32)
SHA32 = _sha(P32)
if not SHA32.startswith("294c1fab84de0bb8"):
    _stop10d("M2ENC_W8A16_FP32_SHA_MISMATCH", f"FP32 ONNX sha={SHA32[:16]} (기대 294c1fab84de0bb8)")
IN_NAME  = onnx.load(str(P32)).graph.input[0].name
OUT_NAME = onnx.load(str(P32)).graph.output[0].name

OUT16 = Path(TRAIN_DIR) / "int8_v1" / "w8a16_10d"
OUT16.mkdir(parents=True, exist_ok=True)
PRE16 = OUT16 / "enc_fp32_pre.onnx"
RAW16 = OUT16 / "enc_w8a16_raw.onnx"
P16   = OUT16 / "enc_w8a16_pow2.onnx"

# ── 1) 보정 데이터: Cell 10의 FULL_TEST 11×200 = 2,200윈도 (재추출 금지)
def _as_np(v):
    try:
        import torch
        if isinstance(v, torch.Tensor):
            return v.detach().cpu().numpy()
    except Exception:
        pass
    return v if isinstance(v, np.ndarray) else None

_hits = []
for _k, _v in list(globals().items()):
    if _k.startswith("_"):
        continue
    _a = _as_np(_v)
    if _a is not None and _a.ndim in (2, 3) and _a.shape[0] == 2200 and _a.shape[-1] == 2048:
        _hits.append(_k)
if len(_hits) != 1:
    _stop10d("M2ENC_W8A16_CAL_NOT_FOUND",
             f"2,200×2048 보정 배열 후보 {_hits} (정확히 1개여야 함) → 화면 캡처 보내주세요")
CAL_NAME = _hits[0]
CAL16 = np.ascontiguousarray(_as_np(globals()[CAL_NAME]).astype(np.float32).reshape(2200, 1, 2048))
if np.abs(CAL16).max() > float(Q_IN) + 1e-6:
    _stop10d("M2ENC_W8A16_CAL_RANGE", f"|max|={np.abs(CAL16).max():.4f} > Q_IN")
CAL_SHA = hashlib.sha256(CAL16.tobytes()).hexdigest()
print(f"[CAL] '{CAL_NAME}' shape={CAL16.shape} |max|={np.abs(CAL16).max():.4f} sha={CAL_SHA[:16]}… (LEARNING 미포함)")

# ── 2) 전처리 + 정적 양자화 (A16W8)
quant_pre_process(str(P32), str(PRE16))

class _R(CalibrationDataReader):
    def __init__(s, X, name): s.X, s.n, s.i = X, name, 0
    def get_next(s):
        if s.i >= len(s.X): return None
        d = {s.n: s.X[s.i:s.i + 1]}; s.i += 1; return d
    def rewind(s): s.i = 0

S_IN16 = np.float32(2.0 ** -12)
t0 = time.time()
quantize_static(
    str(PRE16), str(RAW16), _R(CAL16, IN_NAME),
    quant_format=QuantFormat.QDQ, per_channel=True,
    weight_type=QuantType.QInt8, activation_type=QuantType.QInt16,
    calibrate_method=CalibrationMethod.MinMax,
    extra_options={
        "WeightSymmetric": True,
        "ActivationSymmetric": True,
        "UseQDQContribOps": True,   # opset 17에서 16비트 Q/DQ 사용 (ST 변환 시 opset 21 전환은 별도 단계)
        "TensorQuantOverrides": {IN_NAME: [{
            "scale": np.array(S_IN16, dtype=np.float32),
            "zero_point": np.array(0, dtype=np.int16)}]},
    })
print(f"[QUANT] A16W8 MinMax 완료 {time.time()-t0:.1f}s")

# ── 3) 활성 scale → 2의 거듭제곱(올림, 잘림 방지) + zp=0/int16 검사
m = onnx.load(str(RAW16)); G = m.graph
INIT = {i.name: i for i in G.initializer}
PROD = {o: n for n in G.node for o in n.output}
act_rows, done = [], set()
for n in G.node:
    if n.op_type != "QuantizeLinear" or n.input[0] in INIT:
        continue
    sname = n.input[1]
    zname = n.input[2] if len(n.input) > 2 else None
    if not zname or zname not in INIT:
        _stop10d("M2ENC_W8A16_ZP_MISSING", n.input[0])
    zp = numpy_helper.to_array(INIT[zname])
    if zp.dtype != np.int16 or np.any(zp != 0):
        _stop10d("M2ENC_W8A16_ZP_NOT_ZERO_INT16", f"{n.input[0]} dtype={zp.dtype} val={zp.ravel()[:3]}")
    if sname in done:
        continue
    s_arr = numpy_helper.to_array(INIT[sname]).astype(np.float32)
    s_old = float(s_arr.ravel()[0])
    s_new = float(2.0 ** math.ceil(math.log2(s_old)))
    INIT[sname].CopyFrom(numpy_helper.from_array(np.full(s_arr.shape, s_new, np.float32), sname))
    done.add(sname)
    act_rows.append({"tensor": n.input[0], "scale_old": s_old, "scale_pow2": s_new,
                     "range_pow2": s_new * 32767})

for n in G.node:   # Q/DQ 쌍이 같은 scale을 쓰는지 확인
    if n.op_type == "DequantizeLinear" and n.input[0] in PROD and PROD[n.input[0]].op_type == "QuantizeLinear":
        if n.input[1] != PROD[n.input[0]].input[1]:
            _stop10d("M2ENC_W8A16_QDQ_SCALE_UNPAIRED", PROD[n.input[0]].input[0])

_in = [r for r in act_rows if r["tensor"] == IN_NAME]
if not _in or abs(_in[0]["scale_old"] - float(S_IN16)) > 1e-12:
    _stop10d("M2ENC_W8A16_INPUT_SCALE", f"입력 scale={_in[0]['scale_old'] if _in else None} (목표 2^-12)")

w_rows = []
for n in G.node:
    if n.op_type == "DequantizeLinear" and n.input[0] in INIT:
        q = numpy_helper.to_array(INIT[n.input[0]])
        s = numpy_helper.to_array(INIT[n.input[1]]).astype(np.float64).ravel()
        p2 = bool(np.all(np.abs(np.log2(s) - np.round(np.log2(s))) < 1e-6))
        w_rows.append({"init": n.input[0], "dtype": str(q.dtype), "n_scale": int(s.size), "pow2": p2})

onnx.save(m, str(P16))
try:
    onnx.checker.check_model(str(P16))
except Exception as e:
    print(f"[WARN] onnx checker: {str(e)[:120]} (contrib op 관련이면 무시 가능)")

ops = {}
for n in G.node:
    ops[n.op_type] = ops.get(n.op_type, 0) + 1
print(f"[W8A16] 연산: {ops}")
print(f"[W8A16] 가중치 int8 {sum(r['dtype']=='int8' for r in w_rows)}개 | bias int32 {sum(r['dtype']=='int32' for r in w_rows)}개 "
      f"| 가중치 scale 2의거듭제곱: {sum(r['pow2'] for r in w_rows)}/{len(w_rows)} (참고, ST 변환 단계 확인용)")
print("[W8A16] 활성 scale (원래 → 2의 거듭제곱):")
for r in act_rows:
    print(f"   {r['tensor'][:45]:45s} {r['scale_old']:.3e} → {r['scale_pow2']:.3e}  (범위 ±{r['range_pow2']:.2f})")

# ── 4) 보정 데이터에서 FP32 vs W8A16 참고 점검 (판정 아님)
def _emb(path, X):
    s = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    try:
        return np.concatenate([s.run(None, {IN_NAME: X[i:i+200]})[0] for i in range(0, len(X), 200)])
    except Exception:
        return np.concatenate([s.run(None, {IN_NAME: X[i:i+1]})[0] for i in range(len(X))])

E32, E16 = _emb(P32, CAL16), _emb(P16, CAL16)
cos = (E32 * E16).sum(1) / (np.linalg.norm(E32, axis=1) * np.linalg.norm(E16, axis=1) + 1e-12)
emb_q = PROD[OUT_NAME].input[0] if OUT_NAME in PROD else None
emb_step = next((r["scale_pow2"] for r in act_rows if emb_q and PROD.get(emb_q) is not None
                 and r["tensor"] == PROD[emb_q].input[0]), None)
print(f"[CHECK] 최대 절대오차 {np.abs(E32-E16).max():.3e} | cos 중앙값 {np.median(cos):.5f} 최소 {cos.min():.5f}")
if emb_step:
    print(f"[CHECK] 임베딩 눈금 {emb_step:.2e} vs 등록 최소 std 0.0167 → {0.0167/emb_step:.1f}배 촘촘 (INT8은 0.221)")

# ── 5) Cell 11이 쓸 모델 경로 교체
P8_W8A8_BACKUP = globals().get("P8")
P8 = type(P32)(P16)
SHA16 = _sha(P16)
summary = {"version": "M2_ENC_v1_W8A16_10D_r1", "decision": "D-033",
           "fp32_sha256": SHA32, "w8a16_sha256": SHA16,
           "calibration": {"var": CAL_NAME, "sha256": CAL_SHA, "n": 2200, "learning_used": False},
           "input_scale": float(S_IN16), "activation_dtype": "int16", "weight_dtype": "int8_per_channel",
           "activation_scales": act_rows, "weight_scales_pow2_all": all(r["pow2"] for r in w_rows),
           "ops": ops, "cal_check": {"max_abs_err": float(np.abs(E32-E16).max()),
                                     "cos_median": float(np.median(cos)), "cos_min": float(cos.min())},
           "verdict": None}
(OUT16 / "W8A16_BUILD_SUMMARY.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"[SHA256] w8a16={SHA16[:16]}… | P8 → {P16.name}")
print("STATUS: M2ENC_W8A16_BUILT_AWAITING_EQUIV | LEARNING 사용: 없음 | 다음: Cell 11")


[CAL] 'CAL' shape=(2200, 1, 2048) |max|=8.0000 sha=e62c60c043e999f1… (LEARNING 미포함)


[QUANT] A16W8 MinMax 완료 1.0s
[W8A16] 연산: {'DequantizeLinear': 26, 'QuantizeLinear': 14, 'Conv': 5, 'Relu': 5, 'GlobalAveragePool': 1, 'Flatten': 1, 'Gemm': 1}
[W8A16] 가중치 int8 6개 | bias int32 6개 | 가중치 scale 2의거듭제곱: 0/12 (참고, ST 변환 단계 확인용)
[W8A16] 활성 scale (원래 → 2의 거듭제곱):
   input                                         2.441e-04 → 2.441e-04  (범위 ±8.00)
   /f/f.0/f.0.0/Conv_output_0                    7.265e-04 → 9.766e-04  (범위 ±32.00)
   /f/f.0/f.0.2/Relu_output_0                    7.265e-04 → 9.766e-04  (범위 ±32.00)
   /f/f.1/f.1.0/Conv_output_0                    5.306e-04 → 9.766e-04  (범위 ±32.00)
   /f/f.1/f.1.2/Relu_output_0                    5.306e-04 → 9.766e-04  (범위 ±32.00)
   /f/f.2/f.2.0/Conv_output_0                    4.870e-04 → 4.883e-04  (범위 ±16.00)
   /f/f.2/f.2.2/Relu_output_0                    4.870e-04 → 4.883e-04  (범위 ±16.00)
   /f/f.3/f.3.0/Conv_output_0                    4.166e-04 → 4.883e-04  (범위 ±16.00)
   /f/f.3/f.3.2/Relu_output_0                    4.166e-0

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 10E — w8a16 가중치 pow2 재양자화 후보 (D-036 A 경로)
#  - 원본 enc_w8a16_pow2.onnx 는 읽기만, 결과는 새 폴더(w8a16_10e_w2)에만 저장
#  - 1단계: Q/DQ 도메인 com.microsoft → 표준(opset≥21) 정리 + 출력 '완전 동일' 검증
#  - 2단계: 가중치 scale 2^ceil(log2 s) 로 올림(클리핑 0), 바이어스 scale 동반 보정
#  - LEARNING 사용 없음 / 판정 아님(참고 비교만) / 공식 동등성은 Cell 11
# ══════════════════════════════════════════════════════════════════════
import numpy as np, onnx, hashlib, json, math, shutil
from onnx import numpy_helper, helper
import onnxruntime as ort
from pathlib import Path

def stop(msg): raise RuntimeError(msg)
def sha256p(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def is_pow2(a):
    a = np.atleast_1d(np.asarray(a, dtype=np.float64))
    return bool(np.all(a > 0) and np.all(np.abs(np.log2(a) - np.round(np.log2(a))) < 1e-9))

for v in ["TRAIN_DIR", "CAL"]:
    if v not in globals():
        stop(f"M2ENC_10E_PREREQ_MISSING: {v} 없음 → 셀 1~6, 10, 10B, 10D 순서로 먼저 실행")

INT8_DIR = Path(TRAIN_DIR) / "int8_v1"
SRC      = INT8_DIR / "w8a16_10d" / "enc_w8a16_pow2.onnx"
OUT_DIR  = INT8_DIR / "w8a16_10e_w2"
OUT_DIR.mkdir(parents=True, exist_ok=True)
P_STEP1  = OUT_DIR / "enc_w8a16_std.onnx"        # 도메인 정리본
P_W2     = OUT_DIR / "enc_w8a16_w2pow2.onnx"     # 최종 후보

if not SRC.exists(): stop(f"M2ENC_10E_SRC_NOT_FOUND: {SRC}")
SRC_SHA = sha256p(SRC)
if not SRC_SHA.startswith("9b6b6f528df18813"):
    stop(f"M2ENC_10E_SRC_SHA_MISMATCH: {SRC_SHA[:16]}")
P32_list = [p for p in INT8_DIR.rglob("*.onnx") if sha256p(p).startswith("294c1fab84de0bb8")]
if not P32_list: stop("M2ENC_10E_FP32_NOT_FOUND (sha 294c1fab84de0bb8)")
P32 = P32_list[0]

X = np.asarray(CAL, dtype=np.float32)
if X.ndim == 2 and X.shape[1] == 2048:          # (N, 2048) → (N, 1, 2048) 채널 축만 추가, 값 변경 없음
    X = X[:, None, :]
if X.ndim != 3 or X.shape[1:] != (1, 2048): stop(f"M2ENC_10E_CAL_SHAPE: {X.shape}")
X = np.ascontiguousarray(X)
if X.shape[0] != 2200: stop(f"M2ENC_10E_CAL_COUNT: {X.shape[0]} (10D 기준 2200)")
xmax = float(np.abs(X).max())
if xmax > 8.0 + 1e-6: stop(f"M2ENC_10E_CAL_NOT_CLIPPED: |max|={xmax:.4f} (±8 클립 전 데이터로 보임)")
CAL_SHA = hashlib.sha256(X.tobytes()).hexdigest()
cal_same = CAL_SHA.startswith("e62c60c043e999f1")
print(f"[CAL] {X.shape} |max|={xmax:.4f} sha={CAL_SHA[:16]}… | 10D 보정 데이터와 동일: {cal_same}")
if not cal_same:
    print("⚠ 10D CAL sha와 다름 — 진행은 하지만(참고 비교용) 이 줄을 그대로 보내주세요")
print(f"[SRC] {SRC.name} sha={SRC_SHA[:16]}… | FP32={P32.name} | CAL {X.shape} (FULL_TEST, LEARNING 미포함)")

def run(path, X):
    so = ort.SessionOptions(); so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL
    s = ort.InferenceSession(str(path), so, providers=["CPUExecutionProvider"])
    nin = s.get_inputs()[0].name
    return np.concatenate([s.run(None, {nin: X[i:i+1]})[0] for i in range(len(X))], axis=0)

# ── 1단계: 도메인 정리 ───────────────────────────────────────────────
m = onnx.load(str(SRC))
std_ver = max([o.version for o in m.opset_import if o.domain in ("", "ai.onnx")] or [0])
if std_ver < 21: stop(f"M2ENC_10E_OPSET_LT_21: {std_ver}")
n_dom = 0
for n in m.graph.node:
    if n.op_type in ("QuantizeLinear", "DequantizeLinear") and n.domain == "com.microsoft":
        extra = [a.name for a in n.attribute if a.name != "axis"]
        if extra: stop(f"M2ENC_10E_UNSUPPORTED_ATTR: {n.name} {extra}")
        n.domain = ""; n_dom += 1
others = [n for n in m.graph.node if n.domain == "com.microsoft"]
if others: stop(f"M2ENC_10E_OTHER_MS_OPS: {[n.op_type for n in others]}")
keep = [o for o in m.opset_import if o.domain != "com.microsoft"]
del m.opset_import[:]; m.opset_import.extend(keep)
onnx.checker.check_model(m)
onnx.save(m, str(P_STEP1))

Y_src = run(SRC, X); Y_s1 = run(P_STEP1, X)
step1_exact = bool(np.array_equal(Y_src, Y_s1))
print(f"[STEP1] 도메인 정리 {n_dom}개 노드 | 원본과 출력 완전 동일: {step1_exact}")
if not step1_exact:
    stop(f"M2ENC_10E_STEP1_NOT_IDENTICAL: max|Δ|={np.abs(Y_src-Y_s1).max():.3e} → 출력 보내주세요")

# ── 2단계: 가중치/바이어스 scale pow2 ───────────────────────────────
init = {t.name: t for t in m.graph.initializer}
def arr(name): return numpy_helper.to_array(init[name])
def add_init(a, name):
    t = numpy_helper.from_array(a, name); m.graph.initializer.append(t); init[name] = t; return name
consumers = {}
for n in m.graph.node:
    for i in n.input: consumers.setdefault(i, []).append(n)

ratio_by_wout, wlog = {}, []
for n in m.graph.node:
    if n.op_type != "DequantizeLinear" or n.input[0] not in init: continue
    q = arr(n.input[0])
    if q.dtype != np.int8: continue
    s = arr(n.input[1]).astype(np.float64)
    zp = arr(n.input[2]) if len(n.input) > 2 and n.input[2] in init else np.zeros(1, np.int8)
    if np.any(zp != 0): stop(f"M2ENC_10E_WEIGHT_ZP_NONZERO: {n.name}")
    axis = next((a.i for a in n.attribute if a.name == "axis"), 1)
    ax = axis % q.ndim
    shp = [1]*q.ndim
    if s.size > 1: shp[ax] = s.size
    w  = q.astype(np.float64) * s.reshape(shp)
    s2 = np.power(2.0, np.ceil(np.log2(s)))
    q2 = np.clip(np.round(w / s2.reshape(shp)), -127, 127).astype(np.int8)
    n.input[0] = add_init(q2, n.input[0] + "_w2")
    n.input[1] = add_init(s2.astype(np.float32), n.input[1] + "_w2")
    ratio_by_wout[n.output[0]] = (s2 / s)
    werr = float(np.abs(q2.astype(np.float64)*s2.reshape(shp) - w).max())
    wlog.append(dict(node=n.name, shape=list(q.shape), axis=int(ax), n_ch=int(s.size),
                     ratio_min=float((s2/s).min()), ratio_max=float((s2/s).max()),
                     max_abs_w_err=werr, scale_pow2=is_pow2(s2)))

blog = []
for n in m.graph.node:
    if n.op_type != "DequantizeLinear" or n.input[0] not in init: continue
    qb = arr(n.input[0])
    if qb.dtype != np.int32: continue
    cons = consumers.get(n.output[0], [])
    if len(cons) != 1 or cons[0].op_type not in ("Conv", "Gemm"):
        stop(f"M2ENC_10E_BIAS_CONSUMER_UNEXPECTED: {n.name}")
    wout = cons[0].input[1]
    if wout not in ratio_by_wout: stop(f"M2ENC_10E_BIAS_WEIGHT_LINK_MISSING: {n.name}")
    r  = ratio_by_wout[wout]
    sb = arr(n.input[1]).astype(np.float64)
    sb_full = np.broadcast_to(sb, r.shape).astype(np.float64)
    sb2 = sb_full * r
    qb2 = np.round(qb.astype(np.float64) * sb_full / sb2)
    if np.any(np.abs(qb2) > 2**31 - 1): stop(f"M2ENC_10E_BIAS_OVERFLOW: {n.name}")
    n.input[0] = add_init(qb2.astype(np.int32), n.input[0] + "_w2")
    n.input[1] = add_init(sb2.astype(np.float32), n.input[1] + "_w2")
    if sb.size == 1 and sb2.size > 1:
        for a in list(n.attribute):
            if a.name == "axis": n.attribute.remove(a)
        n.attribute.append(helper.make_attribute("axis", 0))
    blog.append(dict(node=n.name, n_ch=int(sb2.size), scale_pow2=is_pow2(sb2)))

# 사용 안 하게 된 이전 initializer 제거
used = {i for nd in m.graph.node for i in nd.input}
kept = [t for t in m.graph.initializer if t.name in used]
del m.graph.initializer[:]; m.graph.initializer.extend(kept)
onnx.checker.check_model(m)
onnx.save(m, str(P_W2))

# ── 점검: scale 형식 ─────────────────────────────────────────────────
act_scales = []
for n in m.graph.node:
    if n.op_type == "QuantizeLinear" and n.input[1] in {t.name for t in m.graph.initializer}:
        act_scales.append(float(numpy_helper.to_array(next(t for t in m.graph.initializer if t.name == n.input[1]))))
w_pow2 = sum(d["scale_pow2"] for d in wlog); b_pow2 = sum(d["scale_pow2"] for d in blog)
a_pow2 = sum(is_pow2(s) for s in act_scales)
print(f"[W2] 가중치 scale pow2 {w_pow2}/{len(wlog)} | 바이어스 scale pow2 {b_pow2}/{len(blog)} | 활성 scale pow2 {a_pow2}/{len(act_scales)}")
for d in wlog:
    print(f"   {d['node'][:40]:40s} ch={d['n_ch']:3d} ratio {d['ratio_min']:.3f}~{d['ratio_max']:.3f} max|Δw|={d['max_abs_w_err']:.2e}")

# ── 참고 비교 (판정 아님) ────────────────────────────────────────────
Y32 = run(P32, X); Yw2 = run(P_W2, X)
def cos_rows(a, b):
    return np.sum(a*b, 1) / (np.linalg.norm(a, axis=1)*np.linalg.norm(b, axis=1) + 1e-12)
c_src = cos_rows(Y32, Y_src); c_w2 = cos_rows(Y32, Yw2)
print(f"[REF] vs FP32 cos  원본 w8a16: median {np.median(c_src):.5f} / min {c_src.min():.5f}")
print(f"[REF] vs FP32 cos  후보 w2   : median {np.median(c_w2):.5f} / min {c_w2.min():.5f}")
print(f"[REF] vs FP32 max|Δ| 원본 {np.abs(Y32-Y_src).max():.4f} / 후보 {np.abs(Y32-Yw2).max():.4f}")
if c_w2.min() < 0.999: print("⚠ 후보 cos_min < 0.999 — 판정은 Cell 11에서 하지만 출력 그대로 보내주세요")

W2_SHA = sha256p(P_W2)
summary = dict(status="M2ENC_10E_BUILT_AWAITING_ST_CHECK", src=str(SRC), src_sha256=SRC_SHA,
               candidate=str(P_W2), candidate_sha256=W2_SHA, step1_identical=step1_exact,
               weight_pow2=f"{w_pow2}/{len(wlog)}", bias_pow2=f"{b_pow2}/{len(blog)}",
               act_pow2=f"{a_pow2}/{len(act_scales)}",
               ref_cos_vs_fp32=dict(median=float(np.median(c_w2)), min=float(c_w2.min())),
               learning_used=False, weights=wlog, biases=blog)
(OUT_DIR / "W2_SUMMARY.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

DL = Path("/content/m2enc_w8a16_w2_v1.onnx"); shutil.copy(P_W2, DL)
print(f"\n[SHA256] candidate={W2_SHA[:16]}… | 업로드용: {DL}")
print("STATUS: M2ENC_10E_BUILT_AWAITING_ST_CHECK | LEARNING 사용: 없음")
try:
    from google.colab import files; files.download(str(DL))
except Exception as e:
    print("자동 다운로드 실패 → 왼쪽 파일 탭 /content 에서 직접 받으세요:", e)


[CAL] (2200, 1, 2048) |max|=8.0000 sha=e62c60c043e999f1… | 10D 보정 데이터와 동일: True
[SRC] enc_w8a16_pow2.onnx sha=9b6b6f528df18813… | FP32=enc_fp32.onnx | CAL (2200, 1, 2048) (FULL_TEST, LEARNING 미포함)
[STEP1] 도메인 정리 40개 노드 | 원본과 출력 완전 동일: True
[W2] 가중치 scale pow2 6/6 | 바이어스 scale pow2 1/6 | 활성 scale pow2 14/14
   emb.weight_DequantizeLinear              ch= 16 ratio 1.072~1.880 max|Δw|=1.94e-03
   onnx::Conv_52_DequantizeLinear           ch=  4 ratio 1.211~1.576 max|Δw|=3.64e-03
   onnx::Conv_55_DequantizeLinear           ch=  8 ratio 1.041~1.929 max|Δw|=3.88e-03
   onnx::Conv_58_DequantizeLinear           ch= 16 ratio 1.005~1.914 max|Δw|=3.89e-03
   onnx::Conv_61_DequantizeLinear           ch= 32 ratio 1.040~1.957 max|Δw|=1.95e-03
   onnx::Conv_64_DequantizeLinear           ch= 32 ratio 1.008~1.970 max|Δw|=3.90e-03
[REF] vs FP32 cos  원본 w8a16: median 0.99997 / min 0.99982
[REF] vs FP32 cos  후보 w2   : median 0.99994 / min 0.99931
[REF] vs FP32 max|Δ| 원본 0.0828 / 후보 0.3631

[SHA256] candida

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 10F — 바이어스 scale 정합 후보 + w8a8 대조 모델 준비 (D-036 A 경로 후속)
#  - 입력: 10E 후보(enc_w8a16_w2pow2.onnx, sha 11857104…) 읽기만
#  - 수정: 바이어스 scale = 입력활성 scale × 가중치 scale (채널별), int32 값 재계산
#  - 결과: 새 폴더 w8a16_10f_bfix 에만 저장 / LEARNING 사용 없음 / 판정 아님
# ══════════════════════════════════════════════════════════════════════
import numpy as np, onnx, json, shutil
from onnx import numpy_helper, helper
from pathlib import Path

for v in ["P_W2", "X", "P32", "run", "cos_rows", "sha256p", "is_pow2", "INT8_DIR", "stop"]:
    if v not in globals():
        raise RuntimeError(f"M2ENC_10F_PREREQ_MISSING: {v} → Cell 10E 먼저 실행")

W2_SHA_NOW = sha256p(P_W2)
if not W2_SHA_NOW.startswith("11857104f21b93fe"):
    stop(f"M2ENC_10F_W2_SHA_MISMATCH: {W2_SHA_NOW[:16]}")

OUT_F = INT8_DIR / "w8a16_10f_bfix"; OUT_F.mkdir(parents=True, exist_ok=True)
P_BF  = OUT_F / "enc_w8a16_w2_bfix.onnx"

m = onnx.load(str(P_W2))
init = {t.name: t for t in m.graph.initializer}
def A(name): return numpy_helper.to_array(init[name])
def add_init(a, name):
    t = numpy_helper.from_array(a, name); m.graph.initializer.append(t); init[name] = t; return name
prod = {o: n for n in m.graph.node for o in n.output}

def dq_of(tensor):
    n = prod.get(tensor)
    if n is None or n.op_type != "DequantizeLinear": return None
    s  = A(n.input[1]).astype(np.float64).reshape(-1)
    zp = A(n.input[2]).reshape(-1) if len(n.input) > 2 and n.input[2] in init else None
    return n, s, zp

blog = []
for nd in m.graph.node:
    if nd.op_type not in ("Conv", "Gemm") or len(nd.input) < 3: continue
    px, pw, pb = dq_of(nd.input[0]), dq_of(nd.input[1]), dq_of(nd.input[2])
    if not (px and pw and pb): stop(f"M2ENC_10F_QDQ_PATTERN: {nd.name}")
    xn, sx, zx = px; _, sw, _ = pw; bn, sb, _ = pb
    if sx.size != 1: stop(f"M2ENC_10F_ACT_SCALE_NOT_PER_TENSOR: {nd.name}")
    if zx is not None and np.any(zx != 0): stop(f"M2ENC_10F_ACT_ZP_NONZERO: {nd.name}")
    qb = A(bn.input[0]).astype(np.float64).reshape(-1)
    C  = qb.size
    if sw.size not in (1, C) or sb.size not in (1, C): stop(f"M2ENC_10F_CH_MISMATCH: {nd.name}")
    sw_f = np.broadcast_to(sw, (C,)).astype(np.float64)
    sb_f = np.broadcast_to(sb, (C,)).astype(np.float64)
    b_float = qb * sb_f                               # 현재 바이어스 실수값 (보존 대상)
    sb_new  = float(sx[0]) * sw_f                     # 정합 scale = 입력 × 가중치
    ratio_before = sb_f / sb_new                      # 1.0 이면 원래 정합돼 있었음
    qb2 = np.round(b_float / sb_new)
    if np.any(np.abs(qb2) > 2**31 - 1): stop(f"M2ENC_10F_BIAS_OVERFLOW: {nd.name}")
    berr = float(np.abs(qb2 * sb_new - b_float).max())
    bn.input[0] = add_init(qb2.astype(np.int32), bn.input[0] + "_bfix")
    bn.input[1] = add_init(sb_new.astype(np.float32), bn.input[1] + "_bfix")
    if len(bn.input) > 2 and bn.input[2] in init:
        bn.input[2] = add_init(np.zeros(C, np.int32), bn.input[2] + "_bfix")
    for a in list(bn.attribute):
        if a.name == "axis": bn.attribute.remove(a)
    bn.attribute.append(helper.make_attribute("axis", 0))
    blog.append(dict(node=nd.name, op=nd.op_type, ch=C, act_scale=float(sx[0]),
                     act_pow2=is_pow2(sx), ratio_before_min=float(ratio_before.min()),
                     ratio_before_max=float(ratio_before.max()),
                     bias_scale_pow2_after=is_pow2(sb_new), max_abs_bias_err=berr))

used = {i for nd in m.graph.node for i in nd.input}
kept = [t for t in m.graph.initializer if t.name in used]
del m.graph.initializer[:]; m.graph.initializer.extend(kept)
onnx.checker.check_model(m)
onnx.save(m, str(P_BF))

print(f"[BIAS] 대상 {len(blog)}개 층 | 수정 후 바이어스 scale pow2 {sum(d['bias_scale_pow2_after'] for d in blog)}/{len(blog)}")
for d in blog:
    tag = "원래 정합" if abs(d['ratio_before_min']-1)<1e-6 and abs(d['ratio_before_max']-1)<1e-6 else "불일치였음"
    print(f"   {d['op']:4s} {d['node'][:34]:34s} ch={d['ch']:3d} 수정 전 비율 {d['ratio_before_min']:.3f}~{d['ratio_before_max']:.3f} ({tag}) max|Δb|={d['max_abs_bias_err']:.2e}")

# ── 참고 비교 (판정 아님) ────────────────────────────────────────────
Y32 = run(P32, X); Yw2 = run(P_W2, X); Ybf = run(P_BF, X)
c_bf = cos_rows(Y32, Ybf)
print(f"[REF] 후보 bfix vs 10E w2 max|Δ| {np.abs(Ybf-Yw2).max():.2e}  (바이어스 반올림 차이만 있어야 정상)")
print(f"[REF] 후보 bfix vs FP32 cos median {np.median(c_bf):.5f} / min {c_bf.min():.5f} | max|Δ| {np.abs(Y32-Ybf).max():.4f}")

BF_SHA = sha256p(P_BF)
DL_BF = Path("/content/m2enc_w8a16_bfix_v1.onnx"); shutil.copy(P_BF, DL_BF)

# ── w8a8 대조 모델 (예전 V1, sha ee91f648…) — ST에서 Conv HW 매핑 여부 확인용 ──
ctrl = [p for p in INT8_DIR.rglob("*.onnx") if sha256p(p).startswith("ee91f648cc02e369")]
DL_CT = None
if ctrl:
    DL_CT = Path("/content/m2enc_w8a8_ctrl_v1.onnx"); shutil.copy(ctrl[0], DL_CT)
    print(f"[CTRL] w8a8 대조 모델: {ctrl[0].name} → {DL_CT}")
else:
    print("⚠ [CTRL] w8a8(ee91f648…) 파일 못 찾음 — 이 줄 그대로 보내주세요")

summary = dict(status="M2ENC_10F_BUILT_AWAITING_ST_CHECK", src=str(P_W2), src_sha256=W2_SHA_NOW,
               candidate=str(P_BF), candidate_sha256=BF_SHA, biases=blog,
               ref_cos_vs_fp32=dict(median=float(np.median(c_bf)), min=float(c_bf.min())),
               control_w8a8=str(ctrl[0]) if ctrl else None, learning_used=False)
(OUT_F / "BFIX_SUMMARY.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\n[SHA256] bfix={BF_SHA[:16]}…")
print("STATUS: M2ENC_10F_BUILT_AWAITING_ST_CHECK | LEARNING 사용: 없음")
try:
    from google.colab import files
    files.download(str(DL_BF))
    if DL_CT: files.download(str(DL_CT))
except Exception as e:
    print("자동 다운로드 실패 → 왼쪽 파일 탭 /content 에서 직접 받으세요:", e)


[BIAS] 대상 6개 층 | 수정 후 바이어스 scale pow2 6/6
   Conv /f/f.0/f.0.0/Conv                  ch=  4 수정 전 비율 1.000~1.000 (원래 정합) max|Δb|=0.00e+00
   Conv /f/f.1/f.1.0/Conv                  ch=  8 수정 전 비율 0.744~0.744 (불일치였음) max|Δb|=1.84e-06
   Conv /f/f.2/f.2.0/Conv                  ch= 16 수정 전 비율 0.543~0.543 (불일치였음) max|Δb|=3.49e-06
   Conv /f/f.3/f.3.0/Conv                  ch= 32 수정 전 비율 0.997~0.997 (불일치였음) max|Δb|=8.48e-07
   Conv /f/f.4/f.4.0/Conv                  ch= 32 수정 전 비율 0.853~0.853 (불일치였음) max|Δb|=1.64e-06
   Gemm /emb/Gemm                          ch= 16 수정 전 비율 0.805~0.805 (불일치였음) max|Δb|=7.20e-07
[REF] 후보 bfix vs 10E w2 max|Δ| 1.95e-03  (바이어스 반올림 차이만 있어야 정상)
[REF] 후보 bfix vs FP32 cos median 0.99994 / min 0.99931 | max|Δ| 0.3650
[CTRL] w8a8 대조 모델: enc_int8_qdq.onnx → /content/m2enc_w8a8_ctrl_v1.onnx

[SHA256] bfix=bf4acede297a3605…
STATUS: M2ENC_10F_BUILT_AWAITING_ST_CHECK | LEARNING 사용: 없음


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 10G — Conv1D → Conv2D(1×k) 형식 변환 후 재양자화 (D-036 A 경로, NPU 매핑 진단)
#  - 가중치·계산 동일 (재학습 없음). 기존 파일은 읽기만, 결과는 w8a16_10g_2d 에만 저장
#  - w8a8_2d  : NPU 매핑 진단용 (제품 후보 아님)
#  - w8a16_2d : 제품 후보 (활성/가중치 pow2 + 바이어스 정합 = 10D+10E+10F 동일 규칙)
#  - LEARNING 사용 없음 / 판정 아님 / 공식 동등성은 Cell 11
# ══════════════════════════════════════════════════════════════════════
import copy, json, shutil, collections, numpy as np, torch, torch.nn as nn, onnx
from pathlib import Path
from onnx import numpy_helper, helper
from onnxruntime.quantization import (quantize_static, QuantFormat, QuantType,
                                      CalibrationMethod, CalibrationDataReader)

def stop(msg): raise RuntimeError(msg)
need = ["ENC", "X", "P32", "P_BF", "INT8_DIR", "run", "cos_rows", "sha256p", "is_pow2"]
miss = [v for v in need if v not in globals()]
if miss: stop(f"M2ENC_10G_PREREQ_MISSING: {miss} → Cell 10, 10D, 10E, 10F 순서로 먼저 실행")
if not sha256p(P_BF).startswith("bf4acede297a3605"): stop("M2ENC_10G_BFIX_SHA_MISMATCH")
if X.shape != (2200, 1, 2048): stop(f"M2ENC_10G_X_SHAPE: {X.shape}")

OUT_G  = INT8_DIR / "w8a16_10g_2d"; OUT_G.mkdir(parents=True, exist_ok=True)
P32_2D = OUT_G / "enc_fp32_2d.onnx";      P_PRE  = OUT_G / "enc_fp32_2d_pre.onnx"
P8_2D  = OUT_G / "enc_w8a8_2d.onnx"
P16RAW = OUT_G / "enc_w8a16_2d_raw.onnx"; P16_2D = OUT_G / "enc_w8a16_2d_pow2.onnx"
X4 = np.ascontiguousarray(X[:, :, None, :])          # (2200,1,1,2048) — 메모리 배열 동일

# ── 1. PyTorch 1D → 2D 변환 ─────────────────────────────────────────
ENC1 = copy.deepcopy(ENC).cpu().float().eval()
ENC2 = copy.deepcopy(ENC1)
conv_log = []
def to2d(mod, prefix=""):
    for name, ch in list(mod.named_children()):
        full = f"{prefix}.{name}" if prefix else name
        if isinstance(ch, nn.Conv1d):
            if isinstance(ch.padding, str) or ch.padding_mode != "zeros":
                stop(f"M2ENC_10G_CONV_PADDING_UNSUPPORTED: {full} {ch.padding} {ch.padding_mode}")
            c2 = nn.Conv2d(ch.in_channels, ch.out_channels, (1, ch.kernel_size[0]),
                           stride=(1, ch.stride[0]), padding=(0, ch.padding[0]),
                           dilation=(1, ch.dilation[0]), groups=ch.groups, bias=ch.bias is not None)
            with torch.no_grad():
                c2.weight.copy_(ch.weight.unsqueeze(2))
                if ch.bias is not None: c2.bias.copy_(ch.bias)
            setattr(mod, name, c2)
            conv_log.append(f"{full}: Conv1d k={ch.kernel_size[0]} s={ch.stride[0]} p={ch.padding[0]} "
                            f"{ch.in_channels}→{ch.out_channels} ⇒ Conv2d (1,{ch.kernel_size[0]})")
        elif isinstance(ch, nn.BatchNorm1d):
            b2 = nn.BatchNorm2d(ch.num_features, eps=ch.eps, momentum=ch.momentum,
                                affine=ch.affine, track_running_stats=ch.track_running_stats)
            b2.load_state_dict(ch.state_dict()); setattr(mod, name, b2)
        elif isinstance(ch, nn.AdaptiveAvgPool1d):
            o = ch.output_size if isinstance(ch.output_size, int) else ch.output_size[0]
            setattr(mod, name, nn.AdaptiveAvgPool2d((1, o)))
        elif type(ch).__name__.endswith("1d"):
            stop(f"M2ENC_10G_UNSUPPORTED_1D_MODULE: {full} {type(ch).__name__}")
        else:
            to2d(ch, full)
to2d(ENC2); ENC2.eval()
print("[2D] 변환 층:"); [print("   " + s) for s in conv_log]

with torch.no_grad():
    xt = torch.from_numpy(X)
    y1 = torch.cat([ENC1(xt[i:i+256]) for i in range(0, len(xt), 256)]).numpy()
    y2 = torch.cat([ENC2(xt[i:i+256].unsqueeze(2)) for i in range(0, len(xt), 256)]).numpy()
d_t = float(np.abs(y1 - y2).max())
print(f"[2D] PyTorch 1D vs 2D 최대 절대오차 {d_t:.2e} (기준 ≤ 1e-5)")
if d_t > 1e-5: stop("M2ENC_10G_TORCH_2D_MISMATCH → 출력 보내주세요")

# ── 2. FP32 ONNX 내보내기 + 1D ONNX 와 비교 ─────────────────────────
x0 = torch.from_numpy(X4[:1])
kw = dict(input_names=["input"], output_names=["embedding"], opset_version=17, do_constant_folding=True)
try:    torch.onnx.export(ENC2, (x0,), str(P32_2D), dynamo=False, **kw)
except TypeError: torch.onnx.export(ENC2, (x0,), str(P32_2D), **kw)
ops32 = collections.Counter(n.op_type for n in onnx.load(str(P32_2D)).graph.node)
Y32_1d = run(P32, X); Y32_2d = run(P32_2D, X4)
d_o = float(np.abs(Y32_1d - Y32_2d).max())
print(f"[FP32] 2D 연산: {dict(ops32)}")
print(f"[FP32] ONNX 1D vs 2D 최대 절대오차 {d_o:.2e} (기준 ≤ 1e-4)")
if d_o > 1e-4: stop("M2ENC_10G_ONNX_2D_MISMATCH → 출력 보내주세요")
if ops32.get("BatchNormalization", 0): print("⚠ BN 잔존 — 전처리에서 융합 예정, 결과 연산표 확인")

try:
    from onnxruntime.quantization.shape_inference import quant_pre_process
    quant_pre_process(str(P32_2D), str(P_PRE)); SRC_Q = P_PRE
    print(f"[PRE] 전처리 적용 | vs FP32 2D 최대오차 {np.abs(run(P_PRE, X4) - Y32_2d).max():.2e}")
except Exception as e:
    print("[PRE] 전처리 생략:", e); SRC_Q = P32_2D

class CR(CalibrationDataReader):
    def __init__(self, a): self.it = iter([{"input": a[i:i+1]} for i in range(len(a))])
    def get_next(self): return next(self.it, None)

# ── 3. w8a8 (진단용, Cell 10 V1 과 같은 설정) ────────────────────────
quantize_static(str(SRC_Q), str(P8_2D), CR(X4), quant_format=QuantFormat.QDQ, per_channel=True,
    activation_type=QuantType.QInt8, weight_type=QuantType.QInt8,
    calibrate_method=CalibrationMethod.MinMax,
    extra_options={"WeightSymmetric": True, "ActivationSymmetric": False,
        "TensorQuantOverrides": {"input": [{"scale": np.array(8.0/127.0, dtype=np.float32),
                                            "zero_point": np.array(0, dtype=np.int8)}]}})

# ── 4. w8a16 (제품 후보) + pow2/바이어스 정합 ────────────────────────
quantize_static(str(SRC_Q), str(P16RAW), CR(X4), quant_format=QuantFormat.QDQ, per_channel=True,
    activation_type=QuantType.QInt16, weight_type=QuantType.QInt8,
    calibrate_method=CalibrationMethod.MinMax,
    extra_options={"WeightSymmetric": True, "ActivationSymmetric": True,
        "TensorQuantOverrides": {"input": [{"scale": np.array(2.0**-12, dtype=np.float32),
                                            "zero_point": np.array(0, dtype=np.int16)}]}})

def pow2fix(src, dst):
    m = onnx.load(str(src))
    for n in m.graph.node:
        if n.op_type in ("QuantizeLinear", "DequantizeLinear") and n.domain == "com.microsoft": n.domain = ""
    if any(n.domain == "com.microsoft" for n in m.graph.node): stop("M2ENC_10G_OTHER_MS_OPS")
    keep = [o for o in m.opset_import if o.domain != "com.microsoft"]
    del m.opset_import[:]; m.opset_import.extend(keep)
    init = {t.name: t for t in m.graph.initializer}
    A = lambda nm: numpy_helper.to_array(init[nm])
    def put(nm, a): init[nm].CopyFrom(numpy_helper.from_array(a, nm))
    def add(a, nm):
        m.graph.initializer.append(numpy_helper.from_array(a, nm)); init[nm] = m.graph.initializer[-1]; return nm
    # (a) 활성 scale → 2^ceil, zp=0 확인, Q/DQ 공유 확인
    act = {n.input[1] for n in m.graph.node if n.op_type == "QuantizeLinear"}
    for n in m.graph.node:
        if n.op_type == "QuantizeLinear" and len(n.input) > 2 and n.input[2] in init and np.any(A(n.input[2]) != 0):
            stop(f"M2ENC_10G_ACT_ZP_NONZERO: {n.name}")
        if n.op_type == "DequantizeLinear" and n.input[0] not in init and n.input[1] not in act:
            stop(f"M2ENC_10G_ACT_SCALE_NOT_SHARED: {n.name}")
    for nm in act:
        s = A(nm).astype(np.float64)
        put(nm, np.power(2.0, np.ceil(np.log2(s) - 1e-9)).astype(np.float32))
    # (b) 가중치 scale → 2^ceil, 재양자화
    for n in m.graph.node:
        if n.op_type != "DequantizeLinear" or n.input[0] not in init: continue
        q = A(n.input[0])
        if q.dtype != np.int8: continue
        s = A(n.input[1]).astype(np.float64).reshape(-1)
        ax = next((a.i for a in n.attribute if a.name == "axis"), 1) % q.ndim
        shp = [1]*q.ndim
        if s.size > 1: shp[ax] = s.size
        w = q.astype(np.float64) * s.reshape(shp)
        s2 = np.power(2.0, np.ceil(np.log2(s) - 1e-9))
        q2 = np.clip(np.round(w / s2.reshape(shp)), -127, 127).astype(np.int8)
        n.input[0] = add(q2, n.input[0] + "_p2"); n.input[1] = add(s2.astype(np.float32), n.input[1] + "_p2")
    # (c) 바이어스 scale = 입력활성 × 가중치
    prod = {o: n for n in m.graph.node for o in n.output}
    def dqs(t):
        n = prod.get(t)
        return (n, A(n.input[1]).astype(np.float64).reshape(-1)) if n is not None and n.op_type == "DequantizeLinear" else (None, None)
    for nd in m.graph.node:
        if nd.op_type not in ("Conv", "Gemm") or len(nd.input) < 3: continue
        (_, sx), (_, sw), (bn, sb) = dqs(nd.input[0]), dqs(nd.input[1]), dqs(nd.input[2])
        if bn is None or sx is None or sw is None or sx.size != 1: stop(f"M2ENC_10G_QDQ_PATTERN: {nd.name}")
        qb = A(bn.input[0]).astype(np.float64).reshape(-1); C = qb.size
        b_float = qb * np.broadcast_to(sb, (C,))
        sb2 = float(sx[0]) * np.broadcast_to(sw, (C,)).astype(np.float64)
        qb2 = np.round(b_float / sb2)
        if np.any(np.abs(qb2) > 2**31 - 1): stop(f"M2ENC_10G_BIAS_OVERFLOW: {nd.name}")
        bn.input[0] = add(qb2.astype(np.int32), bn.input[0] + "_p2")
        bn.input[1] = add(sb2.astype(np.float32), bn.input[1] + "_p2")
        if len(bn.input) > 2 and bn.input[2] in init: bn.input[2] = add(np.zeros(C, np.int32), bn.input[2] + "_p2")
        for a in list(bn.attribute):
            if a.name == "axis": bn.attribute.remove(a)
        bn.attribute.append(helper.make_attribute("axis", 0))
    used = {i for nd in m.graph.node for i in nd.input}
    kept = [t for t in m.graph.initializer if t.name in used]
    del m.graph.initializer[:]; m.graph.initializer.extend(kept)
    onnx.checker.check_model(m); onnx.save(m, str(dst))

def scale_report(path):
    m = onnx.load(str(path)); init = {t.name: numpy_helper.to_array(t) for t in m.graph.initializer}
    act = {n.input[1] for n in m.graph.node if n.op_type == "QuantizeLinear"}
    w = [n.input[1] for n in m.graph.node if n.op_type == "DequantizeLinear" and n.input[0] in init and init[n.input[0]].dtype == np.int8]
    b = [n.input[1] for n in m.graph.node if n.op_type == "DequantizeLinear" and n.input[0] in init and init[n.input[0]].dtype == np.int32]
    f = lambda L: f"{sum(is_pow2(init[x]) for x in L)}/{len(L)}"
    return dict(act=f(act), weight=f(w), bias=f(b), ops=dict(collections.Counter(n.op_type for n in m.graph.node)))

pow2fix(P16RAW, P16_2D)

# ── 5. 비교 (판정 아님) ──────────────────────────────────────────────
Y16_2d = run(P16_2D, X4); Ybf = run(P_BF, X); Y8_2d = run(P8_2D, X4)
c16 = cos_rows(Y32_1d, Y16_2d)
r16, r8 = scale_report(P16_2D), scale_report(P8_2D)
print(f"\n[W8A16_2D] scale pow2  활성 {r16['act']} | 가중치 {r16['weight']} | 바이어스 {r16['bias']}")
print(f"[W8A16_2D] 연산: {r16['ops']}")
print(f"[W8A16_2D] vs 1D bfix(bf4a…) 최대 절대오차 {np.abs(Y16_2d - Ybf).max():.2e}  (0 또는 매우 작아야 정상)")
print(f"[W8A16_2D] vs FP32 cos median {np.median(c16):.5f} / min {c16.min():.5f}")
ctrl = Path("/content/m2enc_w8a8_ctrl_v1.onnx")
if ctrl.exists():
    print(f"[W8A8_2D ] vs 1D w8a8 대조(ee91…) 최대 절대오차 {np.abs(Y8_2d - run(ctrl, X)).max():.2e}  (참고)")
print(f"[W8A8_2D ] 연산: {r8['ops']}")

S16, S8 = sha256p(P16_2D), sha256p(P8_2D)
json.dump(dict(status="M2ENC_10G_BUILT_AWAITING_ST_CHECK", torch_2d_maxerr=d_t, onnx_2d_maxerr=d_o,
               w8a16_2d=dict(path=str(P16_2D), sha256=S16, **r16,
                             diff_vs_bfix=float(np.abs(Y16_2d - Ybf).max()),
                             cos_vs_fp32=dict(median=float(np.median(c16)), min=float(c16.min()))),
               w8a8_2d=dict(path=str(P8_2D), sha256=S8, purpose="NPU 매핑 진단용, 제품 후보 아님"),
               learning_used=False),
          open(OUT_G / "W2D_SUMMARY.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)

D16, D8 = Path("/content/m2enc_w8a16_2d_v1.onnx"), Path("/content/m2enc_w8a8_2d_v1.onnx")
shutil.copy(P16_2D, D16); shutil.copy(P8_2D, D8)
print(f"\n[SHA256] w8a16_2d={S16[:16]}… | w8a8_2d={S8[:16]}…")
print("STATUS: M2ENC_10G_BUILT_AWAITING_ST_CHECK | LEARNING 사용: 없음")
try:
    from google.colab import files; files.download(str(D8)); files.download(str(D16))
except Exception as e:
    print("자동 다운로드 실패 → 왼쪽 파일 탭 /content 에서 직접 받으세요:", e)


[2D] 변환 층:
   f.0.0: Conv1d k=12 s=4 p=4 1→4 ⇒ Conv2d (1,12)
   f.1.0: Conv1d k=12 s=4 p=4 4→8 ⇒ Conv2d (1,12)
   f.2.0: Conv1d k=6 s=2 p=2 8→16 ⇒ Conv2d (1,6)
   f.3.0: Conv1d k=6 s=2 p=2 16→32 ⇒ Conv2d (1,6)
   f.4.0: Conv1d k=3 s=1 p=1 32→32 ⇒ Conv2d (1,3)
[2D] PyTorch 1D vs 2D 최대 절대오차 0.00e+00 (기준 ≤ 1e-5)


/tmp/ipykernel_2377/398360079.py:71: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  try:    torch.onnx.export(ENC2, (x0,), str(P32_2D), dynamo=False, **kw)


[FP32] 2D 연산: {'Conv': 5, 'Relu': 5, 'GlobalAveragePool': 1, 'Flatten': 1, 'Gemm': 1}
[FP32] ONNX 1D vs 2D 최대 절대오차 0.00e+00 (기준 ≤ 1e-4)
[PRE] 전처리 적용 | vs FP32 2D 최대오차 0.00e+00



[W8A16_2D] scale pow2  활성 13/13 | 가중치 6/6 | 바이어스 6/6
[W8A16_2D] 연산: {'DequantizeLinear': 26, 'QuantizeLinear': 14, 'Conv': 5, 'Relu': 5, 'GlobalAveragePool': 1, 'Flatten': 1, 'Gemm': 1}
[W8A16_2D] vs 1D bfix(bf4a…) 최대 절대오차 1.95e-03  (0 또는 매우 작아야 정상)
[W8A16_2D] vs FP32 cos median 0.99994 / min 0.99931
[W8A8_2D ] vs 1D w8a8 대조(ee91…) 최대 절대오차 0.00e+00  (참고)
[W8A8_2D ] 연산: {'DequantizeLinear': 21, 'QuantizeLinear': 9, 'Conv': 5, 'GlobalAveragePool': 1, 'Flatten': 1, 'Gemm': 1}

[SHA256] w8a16_2d=7af08b2089e39d5b… | w8a8_2d=80bc202c22168901…
STATUS: M2ENC_10G_BUILT_AWAITING_ST_CHECK | LEARNING 사용: 없음


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 10H — 첫 Conv 다상(polyphase) 변환: (1→4, k12, s4, p4) ⇒ (4→4, k3, s1, p1), 입력 [N,4,1,512]
#  - 수학적 등가 변환 (재학습 없음). 결과는 w8a16_10h_poly 에만 저장
#  - w8a8_poly: NPU 매핑 진단용 / w8a16_poly: 제품 후보 (10G와 같은 pow2·바이어스 규칙)
#  - LEARNING 사용 없음 / 판정 아님 / 공식 동등성은 Cell 11
# ══════════════════════════════════════════════════════════════════════
import copy, json, shutil, collections, numpy as np, torch, torch.nn as nn, onnx
from pathlib import Path
from onnxruntime.quantization import quantize_static, QuantFormat, QuantType, CalibrationMethod

def stop(msg): raise RuntimeError(msg)
need = ["ENC2", "X", "X4", "P16_2D", "P32", "INT8_DIR", "run", "cos_rows", "sha256p",
        "pow2fix", "scale_report", "CR"]
miss = [v for v in need if v not in globals()]
if miss: stop(f"M2ENC_10H_PREREQ_MISSING: {miss} → Cell 10G 먼저 실행")
if not sha256p(P16_2D).startswith("7af08b2089e39d5b"): stop("M2ENC_10H_2D_SHA_MISMATCH")

OUT_H  = INT8_DIR / "w8a16_10h_poly"; OUT_H.mkdir(parents=True, exist_ok=True)
P32_P  = OUT_H / "enc_fp32_poly.onnx"; P_PREP = OUT_H / "enc_fp32_poly_pre.onnx"
P8_P   = OUT_H / "enc_w8a8_poly.onnx"
P16R_P = OUT_H / "enc_w8a16_poly_raw.onnx"; P16_P = OUT_H / "enc_w8a16_poly_pow2.onnx"
PH = 4                                               # 다상 수 = 첫 층 stride

def to_poly(a):                                      # (N,1,2048) → (N,4,1,512), X[p][m] = x[4m+p]
    N = a.shape[0]
    return np.ascontiguousarray(a.reshape(N, 2048 // PH, PH).transpose(0, 2, 1)[:, :, None, :])
XP = to_poly(X)

# ── 1. 첫 Conv 찾기 + 조건 확인 ─────────────────────────────────────
ENC3 = copy.deepcopy(ENC2).cpu().float().eval()
first_name, c0 = next((n, m) for n, m in ENC3.named_modules() if isinstance(m, nn.Conv2d))
cond = (c0.in_channels == 1 and c0.kernel_size == (1, 12) and c0.stride == (1, 4)
        and c0.padding == (0, 4) and c0.dilation == (1, 1) and c0.groups == 1)
if not cond: stop(f"M2ENC_10H_FIRST_CONV_UNEXPECTED: {first_name} k={c0.kernel_size} s={c0.stride} p={c0.padding}")
K3 = c0.kernel_size[1] // PH                         # 3
P3 = c0.padding[1] // PH                             # 1
cp = nn.Conv2d(PH, c0.out_channels, (1, K3), stride=1, padding=(0, P3), bias=c0.bias is not None)
with torch.no_grad():
    W = c0.weight[:, 0, 0, :]                        # [O, 12]
    cp.weight.copy_(W.reshape(c0.out_channels, K3, PH).permute(0, 2, 1).unsqueeze(2))  # [O,4,1,3]
    if c0.bias is not None: cp.bias.copy_(c0.bias)
parent = ENC3.get_submodule(first_name.rsplit(".", 1)[0]) if "." in first_name else ENC3
setattr(parent, first_name.rsplit(".", 1)[-1], cp)
print(f"[POLY] {first_name}: Conv2d 1→{c0.out_channels} (1,12) s4 p4 ⇒ Conv2d {PH}→{c0.out_channels} (1,{K3}) s1 p{P3} | 입력 {XP.shape[1:]}")

with torch.no_grad():
    y2 = torch.cat([ENC2(torch.from_numpy(X4[i:i+256])) for i in range(0, len(X4), 256)]).numpy()
    y3 = torch.cat([ENC3(torch.from_numpy(XP[i:i+256])) for i in range(0, len(XP), 256)]).numpy()
d_t = float(np.abs(y2 - y3).max())
print(f"[POLY] PyTorch 2D vs 다상 최대 절대오차 {d_t:.2e} (기준 ≤ 1e-5, 덧셈 순서 차이만)")
if d_t > 1e-5: stop("M2ENC_10H_TORCH_POLY_MISMATCH → 출력 보내주세요")

# ── 2. FP32 ONNX ────────────────────────────────────────────────────
kw = dict(input_names=["input"], output_names=["embedding"], opset_version=17, do_constant_folding=True)
x0 = torch.from_numpy(XP[:1])
try:    torch.onnx.export(ENC3, (x0,), str(P32_P), dynamo=False, **kw)
except TypeError: torch.onnx.export(ENC3, (x0,), str(P32_P), **kw)
Y32_1d = run(P32, X); Y32_p = run(P32_P, XP)
d_o = float(np.abs(Y32_1d - Y32_p).max())
print(f"[FP32] 연산: {dict(collections.Counter(n.op_type for n in onnx.load(str(P32_P)).graph.node))}")
print(f"[FP32] ONNX 1D(원본) vs 다상 최대 절대오차 {d_o:.2e} (기준 ≤ 1e-4)")
if d_o > 1e-4: stop("M2ENC_10H_ONNX_POLY_MISMATCH → 출력 보내주세요")
try:
    from onnxruntime.quantization.shape_inference import quant_pre_process
    quant_pre_process(str(P32_P), str(P_PREP)); SRC_Q = P_PREP
except Exception as e:
    print("[PRE] 전처리 생략:", e); SRC_Q = P32_P

# ── 3. 양자화 (10G와 동일 설정) ──────────────────────────────────────
quantize_static(str(SRC_Q), str(P8_P), CR(XP), quant_format=QuantFormat.QDQ, per_channel=True,
    activation_type=QuantType.QInt8, weight_type=QuantType.QInt8, calibrate_method=CalibrationMethod.MinMax,
    extra_options={"WeightSymmetric": True, "ActivationSymmetric": False,
        "TensorQuantOverrides": {"input": [{"scale": np.array(8.0/127.0, dtype=np.float32),
                                            "zero_point": np.array(0, dtype=np.int8)}]}})
quantize_static(str(SRC_Q), str(P16R_P), CR(XP), quant_format=QuantFormat.QDQ, per_channel=True,
    activation_type=QuantType.QInt16, weight_type=QuantType.QInt8, calibrate_method=CalibrationMethod.MinMax,
    extra_options={"WeightSymmetric": True, "ActivationSymmetric": True,
        "TensorQuantOverrides": {"input": [{"scale": np.array(2.0**-12, dtype=np.float32),
                                            "zero_point": np.array(0, dtype=np.int16)}]}})
pow2fix(P16R_P, P16_P)

# ── 4. 비교 (판정 아님) ──────────────────────────────────────────────
Y16_p = run(P16_P, XP); Y16_2d = run(P16_2D, X4)
d_q = float(np.abs(Y16_p - Y16_2d).max()); c16 = cos_rows(Y32_1d, Y16_p)
r16 = scale_report(P16_P)
print(f"\n[W8A16_POLY] scale pow2  활성 {r16['act']} | 가중치 {r16['weight']} | 바이어스 {r16['bias']}")
print(f"[W8A16_POLY] 연산: {r16['ops']}")
print(f"[W8A16_POLY] vs 10G w8a16_2d(7af0…) 최대 절대오차 {d_q:.2e}  (0 또는 1칸=1.95e-03 이하 정상)")
print(f"[W8A16_POLY] vs FP32 cos median {np.median(c16):.5f} / min {c16.min():.5f}")
if d_q > 4e-3: print("⚠ 10G 대비 2칸 이상 차이 — 진행은 가능하지만 이 줄 그대로 보내주세요")

S16, S8 = sha256p(P16_P), sha256p(P8_P)
json.dump(dict(status="M2ENC_10H_BUILT_AWAITING_ST_CHECK", input_layout="[N,4,1,512], X[p][m]=x[4m+p]",
               torch_maxerr=d_t, onnx_maxerr=d_o, diff_vs_10g=d_q,
               w8a16_poly=dict(path=str(P16_P), sha256=S16, **r16,
                               cos_vs_fp32=dict(median=float(np.median(c16)), min=float(c16.min()))),
               w8a8_poly=dict(path=str(P8_P), sha256=S8, purpose="NPU 매핑 진단용"), learning_used=False),
          open(OUT_H / "POLY_SUMMARY.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
D16, D8 = Path("/content/m2enc_w8a16_poly_v1.onnx"), Path("/content/m2enc_w8a8_poly_v1.onnx")
shutil.copy(P16_P, D16); shutil.copy(P8_P, D8)
print(f"\n[SHA256] w8a16_poly={S16[:16]}… | w8a8_poly={S8[:16]}…")
print("STATUS: M2ENC_10H_BUILT_AWAITING_ST_CHECK | LEARNING 사용: 없음")
try:
    from google.colab import files; files.download(str(D16)); files.download(str(D8))
except Exception as e:
    print("자동 다운로드 실패 → 왼쪽 파일 탭 /content 에서 직접 받으세요:", e)


[POLY] f.0.0: Conv2d 1→4 (1,12) s4 p4 ⇒ Conv2d 4→4 (1,3) s1 p1 | 입력 (4, 1, 512)
[POLY] PyTorch 2D vs 다상 최대 절대오차 0.00e+00 (기준 ≤ 1e-5, 덧셈 순서 차이만)


/tmp/ipykernel_2377/2430561514.py:56: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  try:    torch.onnx.export(ENC3, (x0,), str(P32_P), dynamo=False, **kw)


[FP32] 연산: {'Conv': 5, 'Relu': 5, 'GlobalAveragePool': 1, 'Flatten': 1, 'Gemm': 1}
[FP32] ONNX 1D(원본) vs 다상 최대 절대오차 7.63e-06 (기준 ≤ 1e-4)



[W8A16_POLY] scale pow2  활성 13/13 | 가중치 6/6 | 바이어스 6/6
[W8A16_POLY] 연산: {'DequantizeLinear': 26, 'QuantizeLinear': 14, 'Conv': 5, 'Relu': 5, 'GlobalAveragePool': 1, 'Flatten': 1, 'Gemm': 1}
[W8A16_POLY] vs 10G w8a16_2d(7af0…) 최대 절대오차 1.95e-03  (0 또는 1칸=1.95e-03 이하 정상)
[W8A16_POLY] vs FP32 cos median 0.99994 / min 0.99931

[SHA256] w8a16_poly=df6e27f6945c2f66… | w8a8_poly=12e33ef2e7ad0842…
STATUS: M2ENC_10H_BUILT_AWAITING_ST_CHECK | LEARNING 사용: 없음


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ===== M2_ENC_v1 Cell 11 (r2): w8a16_poly 동등성 검사 (D-032 재적용) — FULL_TEST 먼저, 통과 시에만 LEARNING =====
# 기준(D-032/D-021, 변경 없음): FP32 ONNX(1D) 대비 검출 집합 동일 + 레코드별 onset 차이 ≤ 5윈도.
# 코사인·lead·reg_FAR·D-028·시도1 대비 차이는 보고만. LEARNING 읽기 직전에 시도 1회 기록(최대 3회, 공유 파일).
# 쓰기: TRAIN_DIR/int8_v1/ 안만. LEARNING_EVAL.lock 미접촉. Cell 10~10H 재실행 불필요(Cell 1~6만 필요).
import sys, json, hashlib, datetime, subprocess
from pathlib import Path
import numpy as np, pandas as pd

CAND_TAG          = "w8a16_poly_v1"
CAND_SHA_PREFIX   = "df6e27f6945c2f66"   # Cell 10H 출력 = ST 벤치마크 CSV
FP32_SHA_PREFIX   = "294c1fab84de0bb8"   # enc_fp32.onnx (D-032 기준 모델, 1D)
FINAL_SHA_PREFIX  = "508ad40a3fee3ee3"   # FINAL_LIN_D16.pt
ONSET_TOL, MAX_ATTEMPTS = 5, 3
SANITY_N, SANITY_COS_MIN = 256, 0.995

# ── 0. 사전 조건 ──
if "stop" not in globals():
    raise RuntimeError("M2ENC_EQ_PREREQ_MISSING: stop — Cell 1부터 실행")
_req = ["TRAIN_DIR", "run_detector", "DATA", "RECORDS", "lp6000", "load_record", "Q_IN", "REG_MIN"]
_miss = [n for n in _req if n not in globals()]
if _miss: stop("M2ENC_EQ_PREREQ_MISSING", missing=_miss, note="Cell 1~6 먼저 실행")

try:
    import onnxruntime as ort
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "onnxruntime"])
    import onnxruntime as ort
print(f"[ENV] onnxruntime {ort.__version__}")

def _sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()

INT8_DIR = Path(TRAIN_DIR) / "int8_v1"
if not INT8_DIR.exists(): stop("M2ENC_EQ_DIR_NOT_FOUND", path=str(INT8_DIR))

# ── 1. 모델 파일 확인 (SHA로 찾기) ──
_pool = sorted(INT8_DIR.rglob("*.onnx")) + sorted(Path("/content").glob("*.onnx"))
_hits = [p for p in _pool if _sha256(p).startswith(CAND_SHA_PREFIX)]
if not _hits: stop("M2ENC_EQ_CAND_NOT_FOUND", sha_prefix=CAND_SHA_PREFIX, note="Cell 10H 산출물 위치 확인")
P_CAND = _hits[0]; CAND_SHA = _sha256(P_CAND)

P_FP32 = Path(globals().get("P32", INT8_DIR / "enc_fp32.onnx"))
if not P_FP32.exists(): stop("M2ENC_EQ_FP32_NOT_FOUND", path=str(P_FP32))
FP32_SHA = _sha256(P_FP32)
if not FP32_SHA.startswith(FP32_SHA_PREFIX): stop("M2ENC_EQ_FP32_SHA_MISMATCH", got=FP32_SHA[:16])

if "FINAL_SHA" not in globals():
    FINAL_SHA = _sha256(Path(TRAIN_DIR) / "ckpt" / "FINAL_LIN_D16.pt")
if not FINAL_SHA.startswith(FINAL_SHA_PREFIX): stop("M2ENC_EQ_FINAL_SHA_MISMATCH", got=FINAL_SHA[:16])
print(f"[MODEL] 후보 {P_CAND}  sha={CAND_SHA[:16]}…")
print(f"[MODEL] 기준 {P_FP32.name}  sha={FP32_SHA[:16]}… | FINAL sha={FINAL_SHA[:16]}…")

# ── 2. 세션 + 입력 배치 ──
def _layout(shape):
    tail = tuple(d if isinstance(d, int) else None for d in shape[1:])
    if tail == (1, 2048):    return "1D"
    if tail == (1, 1, 2048): return "2D"
    if tail == (4, 1, 512):  return "POLY"
    stop("M2ENC_EQ_INPUT_LAYOUT_UNKNOWN", shape=[str(d) for d in shape])

def _sess(p):
    s = ort.InferenceSession(str(p), providers=["CPUExecutionProvider"])
    i, o = s.get_inputs()[0], s.get_outputs()[0]
    return dict(sess=s, iname=i.name, oname=o.name, shape=list(i.shape), layout=_layout(i.shape),
                fixed1=(i.shape[0] == 1))

def _prep(Z, layout, wrong=False):
    Z = np.asarray(Z, dtype=np.float32); Z = Z.reshape(Z.shape[0], -1)
    if Z.shape[1] != 2048: stop("M2ENC_EQ_WINDOW_LEN", got=int(Z.shape[1]))
    N = Z.shape[0]
    if layout == "1D": return Z[:, None, :]
    if layout == "2D": return Z[:, None, None, :]
    if wrong:          return Z.reshape(N, 4, 512)[:, :, None, :]                 # 점검용(틀린 배열)
    return Z.reshape(N, 512, 4).transpose(0, 2, 1)[:, :, None, :]               # 채널 p, 위치 j = x[4j+p]

def _embed(M, Z, wrong=False, bs=512):
    X = np.ascontiguousarray(_prep(Z, M["layout"], wrong))
    step = 1 if M["fixed1"] else bs
    out = [M["sess"].run([M["oname"]], {M["iname"]: X[i:i + step]})[0] for i in range(0, len(X), step)]
    return np.concatenate(out, 0).reshape(len(X), -1).astype(np.float64)

def _cos(A, B):
    return (A * B).sum(1) / (np.linalg.norm(A, axis=1) * np.linalg.norm(B, axis=1) + 1e-12)

M32, MC = _sess(P_FP32), _sess(P_CAND)
print(f"[INPUT] 기준 {M32['shape']} ({M32['layout']}) | 후보 {MC['shape']} ({MC['layout']})")
if MC["layout"] != "POLY": stop("M2ENC_EQ_CAND_NOT_POLY", shape=[str(d) for d in MC["shape"]])

# ── 3. 입력 변환 점검 (FULL_TEST 256윈도, 시도 소모 없음) ──
_rng = np.random.default_rng(20261030)
_allZ = np.concatenate([np.asarray(DATA[k]["Z"], np.float32).reshape(-1, 2048) for k in sorted(DATA)], 0)
_S = _allZ[np.sort(_rng.choice(len(_allZ), SANITY_N, replace=False))]; del _allZ
_e32 = _embed(M32, _S)
_c_ok, _c_bad = _cos(_e32, _embed(MC, _S)), _cos(_e32, _embed(MC, _S, wrong=True))
print(f"[SANITY] 다상 배열 x[4j+p] cos min {_c_ok.min():.5f} (median {np.median(_c_ok):.5f}) | "
      f"틀린 배열 cos median {np.median(_c_bad):.5f}")
if not (_c_ok.min() >= SANITY_COS_MIN and np.median(_c_ok) > np.median(_c_bad)):
    stop("M2ENC_EQ_POLY_TRANSFORM_MISMATCH", note="입력 배열 불일치 — LEARNING 미사용. 이 출력 그대로 보내주세요")
print("[SANITY] 입력 변환 확인 ✅")

# ── 4. 비교 함수 ──
def _cmp(key, Z, n):
    E32, EC = _embed(M32, Z), _embed(MC, Z)
    r32, rc = run_detector(E32), run_detector(EC)
    cos = _cos(E32, EC)
    same = r32["detected"] == rc["detected"]
    d = abs(r32["onset"] - rc["onset"]) if (r32["detected"] and rc["detected"]) else None
    lead = lambda r: (n - r["onset"]) * 10 / 3600 if r["detected"] else np.nan
    return dict(record_key=key, n=n, det_fp32=r32["detected"], det_cand=rc["detected"],
                onset_fp32=r32["onset"], onset_cand=rc["onset"], onset_diff=d, det_same=same,
                onset_ok=bool(same and (d is None or d <= ONSET_TOL)),
                lead_fp32=lead(r32), lead_cand=lead(rc),
                far_fp32_pct=100 * r32["reg_far"], far_cand_pct=100 * rc["reg_far"],
                cos_median=float(np.median(cos)), cos_min=float(cos.min()))

pd.set_option("display.width", 260)
COLS = ["record_key", "det_fp32", "det_cand", "onset_fp32", "onset_cand", "onset_diff", "onset_ok",
        "lead_fp32", "lead_cand", "far_fp32_pct", "far_cand_pct", "cos_median", "cos_min"]

# ── 5. FULL_TEST 11개 (시도 소모 없음) ──
FT = pd.DataFrame([_cmp(k, DATA[k]["Z"], DATA[k]["n"]) for k in sorted(DATA)])
FT.to_csv(INT8_DIR / f"equiv_fulltest_{CAND_TAG}_{CAND_SHA[:12]}.csv", index=False, encoding="utf-8-sig")
FT_PASS = bool(FT.onset_ok.all())
print(f"\n[FULL_TEST] FP32 ONNX vs {CAND_TAG}\n", FT[COLS].round(4).to_string(index=False))
print(f"[FULL_TEST] 동등성 → {'통과 ✅' if FT_PASS else '실패 ❌'}")
if not FT_PASS:
    (INT8_DIR / f"EQUIV_FULLTEST_FAIL_{CAND_TAG}_{CAND_SHA[:12]}.json").write_text(json.dumps(dict(
        status="M2ENC_POLY_FULLTEST_FAIL", cand_tag=CAND_TAG, cand_sha256=CAND_SHA, learning_read=False,
        failed=FT[~FT.onset_ok].record_key.tolist()), ensure_ascii=False, indent=2), encoding="utf-8")
    stop("M2ENC_POLY_FULLTEST_FAIL", note="LEARNING 미사용 — 시도 소모 없음. 이 출력 그대로 보내주세요")

# ── 6. 시도 기록 (LEARNING 읽기 전, 공유 파일) ──
ATT = INT8_DIR / "INT8_EQUIV_ATTEMPTS.json"
att = json.loads(ATT.read_text(encoding="utf-8")) if ATT.exists() else []
_done = [a for a in att if a.get("int8_sha256") == CAND_SHA]
if _done:
    print(json.dumps(_done, ensure_ascii=False, indent=2, default=str))
    stop("M2ENC_POLY_ALREADY_TESTED", note="같은 모델 재검사 금지 — 위 기록 사용")
if len(att) >= MAX_ATTEMPTS: stop("M2ENC_EQ_ATTEMPTS_EXHAUSTED", note="3회 소진 — 대표 결정 필요")
K_ATT = len(att) + 1
att.append(dict(attempt=K_ATT, started=datetime.datetime.now().isoformat(), model_tag=CAND_TAG,
                int8_sha256=CAND_SHA, fp32_sha256=FP32_SHA, final_sha256=FINAL_SHA,
                input_layout="POLY [N,4,1,512] ch p = x[4j+p]", fulltest_pass=True, status="STARTED"))
ATT.write_text(json.dumps(att, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\n[ATTEMPT] {K_ATT}/{MAX_ATTEMPTS} 기록 — 지금부터 LEARNING 읽기 (중단돼도 1회 소모)")

# ── 7. LEARNING 6개 (Cell 9와 같은 전처리: LP6000 → x/s → clip ±Q) ──
_ev = RECORDS[RECORDS.role == "EVAL"]
if len(_ev) != 6: stop("M2ENC_EQ_EVAL_COUNT", got=int(len(_ev)))
lr_rows = []
for r in _ev.itertuples():
    X = load_record(r); Y = lp6000(X); del X
    s = float(np.median(np.sqrt((Y ** 2).mean(axis=1))[:REG_MIN]))
    Z = np.clip(Y / s, -Q_IN, Q_IN).astype(np.float32); del Y
    lr_rows.append({**_cmp(r.record_key, Z, r.n), "bearing": r.bearing}); del Z
    print(f"  {r.record_key:22s} 완료")
LR = pd.DataFrame(lr_rows)

# 보고 전용: D-031 저장값(GPU) / 시도1 w8a16 원본 onset
_num = lambda s: pd.to_numeric(s, errors="coerce")
_ref = pd.read_csv(Path(TRAIN_DIR) / "learning_eval_once" / "learning_eval_m2.csv")
LR["onset_d031"] = LR.bearing.map(dict(zip(_ref.bearing, _num(_ref.onset))))
LR["fp32_eq_d031"] = _num(LR.onset_fp32) == LR.onset_d031
_a1 = INT8_DIR / "int8_equiv_learning_attempt1.csv"
if _a1.exists():
    _p = pd.read_csv(_a1); LR["onset_att1"] = LR.record_key.map(dict(zip(_p.record_key, _num(_p.onset_int8))))
else:
    LR["onset_att1"] = np.nan
LR.to_csv(INT8_DIR / f"equiv_learning_attempt{K_ATT}_{CAND_TAG}.csv", index=False, encoding="utf-8-sig")

LR_PASS = bool(LR.onset_ok.all())
d028 = dict(detected=int(LR.det_cand.sum()), mean_lead_h=float(LR.lead_cand.mean()),
            mean_reg_far_pct=float(LR.far_cand_pct.mean()))
STATUS = "M2ENC_POLY_EQUIV_PASS" if (FT_PASS and LR_PASS) else "M2ENC_POLY_EQUIV_FAIL"
print(f"\n[LEARNING] FP32 ONNX vs {CAND_TAG}\n",
      LR[COLS + ["onset_d031", "fp32_eq_d031", "onset_att1"]].round(4).to_string(index=False))
print(f"[LEARNING] 동등성 → {'통과 ✅' if LR_PASS else '실패 ❌'}")
print(f"[보고] D-028 수치: 검출 {d028['detected']}/6 | 평균 lead {d028['mean_lead_h']:.4f}h | "
      f"평균 reg_FAR {d028['mean_reg_far_pct']:.4f}%")
print(f"[보고] FP32 ONNX(CPU) = D-031(GPU) onset 일치: {int(LR.fp32_eq_d031.sum())}/6"
      + ("" if LR.fp32_eq_d031.all() else "  ⚠ [확인 필요] 장치 차이"))
print(f"[보고] 코사인 중앙값(최소): FULL_TEST {FT.cos_median.median():.5f} ({FT.cos_min.min():.5f}) / "
      f"LEARNING {LR.cos_median.median():.5f} ({LR.cos_min.min():.5f})")

att[-1].update(status=STATUS, finished=datetime.datetime.now().isoformat(), learning_pass=LR_PASS,
               d028=d028, fp32_eq_d031=int(LR.fp32_eq_d031.sum()))
ATT.write_text(json.dumps(att, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
(INT8_DIR / f"EQUIV_SUMMARY_attempt{K_ATT}_{CAND_TAG}.json").write_text(json.dumps(dict(
    status=STATUS, attempt=K_ATT, max_attempts=MAX_ATTEMPTS, model_tag=CAND_TAG, cand_sha256=CAND_SHA,
    fp32_sha256=FP32_SHA, final_sha256=FINAL_SHA, input_layout="[N,4,1,512], ch p = x[4j+p]",
    sanity=dict(cos_min=float(_c_ok.min()), wrong_layout_cos_median=float(np.median(_c_bad))),
    criteria=dict(detected_set_same=True, onset_diff_max=ONSET_TOL), fulltest_pass=FT_PASS, learning_pass=LR_PASS,
    report_only=dict(d028=d028, fp32_eq_d031=int(LR.fp32_eq_d031.sum()),
                     cos_median_ft=float(FT.cos_median.median()), cos_median_lr=float(LR.cos_median.median())),
    d031_result_unchanged=True, learning_eval_lock_touched=False), ensure_ascii=False, indent=2, default=str),
    encoding="utf-8")
print(f"\nSTATUS: {STATUS} | 시도 {K_ATT}/{MAX_ATTEMPTS} | D-031 결과 불변 | LEARNING_EVAL.lock 미접촉")



[ENV] onnxruntime 1.31.1
[MODEL] 후보 /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/bridge_outputs/m2_enc_v1_train_42f861b9506e/int8_v1/w8a16_10h_poly/enc_w8a16_poly_pow2.onnx  sha=df6e27f6945c2f66…
[MODEL] 기준 enc_fp32.onnx  sha=294c1fab84de0bb8… | FINAL sha=508ad40a3fee3ee3…
[INPUT] 기준 [1, 1, 2048] (1D) | 후보 [1, 4, 1, 512] (POLY)
[SANITY] 다상 배열 x[4j+p] cos min 0.99931 (median 0.99994) | 틀린 배열 cos median 0.78654
[SANITY] 입력 변환 확인 ✅

[FULL_TEST] FP32 ONNX vs w8a16_poly_v1
           record_key  det_fp32  det_cand  onset_fp32  onset_cand  onset_diff  onset_ok  lead_fp32  lead_cand  far_fp32_pct  far_cand_pct  cos_median  cos_min
FULL_TEST/Bearing1_3      True      True      1226.0      1226.0         0.0      True     3.1917     3.1917        0.1686        0.1686      0.9999   0.9996
FULL_TEST/Bearing1_4      True      True       593.0       593.0         0.0      True     2.3194     2.3194        0.0000        0.0000      0.9999   0.9995
FULL_TEST/Bearing1_5      True      True    

In [ ]:
# ===== M2_ENC_v1 Cell 12 (r1): HIL 검증 데이터 생성 (UNRESOLVED #15/#16) — FULL_TEST만, LEARNING 미사용 =====
# 쓰기: TRAIN_DIR/int8_v1/hil_v1/ + /content 복사본만. 모델·동결 파일·LEARNING_EVAL.lock 미접촉. 시도 소모 없음.
import json, hashlib, shutil, datetime, zipfile
from pathlib import Path
import numpy as np, pandas as pd
import onnxruntime as ort

CAND_SHA_PREFIX = "df6e27f6945c2f66"   # D-036 확정 모델
FP32_SHA_PREFIX = "294c1fab84de0bb8"
PER_SEG = 10                            # 등록/중간/말기 각 10윈도 → 레코드당 최대 30
S_IN = 2.0 ** -12                       # D-033 입력 scale
SEED = 20261030

if "stop" not in globals(): raise RuntimeError("M2ENC_HIL_PREREQ_MISSING: Cell 1부터 실행")
_miss = [n for n in ["TRAIN_DIR", "DATA", "Q_IN", "REG_MIN"] if n not in globals()]
if _miss: stop("M2ENC_HIL_PREREQ_MISSING", missing=_miss, note="Cell 1~6 먼저 실행")

def _sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()

INT8_DIR = Path(TRAIN_DIR) / "int8_v1"
_hits = [p for p in sorted(INT8_DIR.rglob("*.onnx")) if _sha256(p).startswith(CAND_SHA_PREFIX)]
if not _hits: stop("M2ENC_HIL_CAND_NOT_FOUND", sha_prefix=CAND_SHA_PREFIX)
P_CAND = _hits[0]; CAND_SHA = _sha256(P_CAND)
P_FP32 = INT8_DIR / "enc_fp32.onnx"
if not (P_FP32.exists() and _sha256(P_FP32).startswith(FP32_SHA_PREFIX)): stop("M2ENC_HIL_FP32_SHA_MISMATCH")
print(f"[MODEL] {P_CAND.name} sha={CAND_SHA[:16]}… (D-036)")

def _sess(p):
    s = ort.InferenceSession(str(p), providers=["CPUExecutionProvider"])
    return s, s.get_inputs()[0].name, s.get_outputs()[0].name
SC, IC, OC = _sess(P_CAND); S32, I32, O32 = _sess(P_FP32)
def _run(s, i, o, X):  # 배치 1 고정 모델 대응
    return np.concatenate([s.run([o], {i: X[k:k + 1]})[0].reshape(1, -1) for k in range(len(X))], 0)

# ── 1. 윈도 선택 (FULL_TEST, 결정적) ──
rows, Zs = [], []
for key in sorted(DATA):
    Z = np.asarray(DATA[key]["Z"], np.float32).reshape(-1, 2048); n = len(Z)
    segs = {"REG":  np.linspace(0, min(REG_MIN, n) - 1, PER_SEG),
            "MID":  np.linspace(0.4 * n, 0.6 * n, PER_SEG),
            "LATE": np.linspace(0.9 * n, n - 1, PER_SEG)}
    seen = set()
    for seg, idx in segs.items():
        for w in np.unique(np.round(idx).astype(int)):
            if w in seen: continue
            seen.add(w); rows.append(dict(record_key=key, window=int(w), n=n, life_frac=round(w / n, 4), seg=seg))
            Zs.append(Z[w])
IDX = pd.DataFrame(rows); Zh = np.stack(Zs).astype(np.float32); N = len(Zh)
zmax = float(np.abs(Zh).max())
if zmax > Q_IN + 1e-6: stop("M2ENC_HIL_NOT_CLIPPED", max=zmax)
print(f"[DATA] FULL_TEST {IDX.record_key.nunique()}개 레코드, {N}윈도 | |max|={zmax:.4f} | LEARNING 미사용")

# ── 2. 입력 배열 (모델 형식 + 펌웨어 형식 2종) ──
X_poly = np.ascontiguousarray(Zh.reshape(N, 512, 4).transpose(0, 2, 1)[:, :, None, :])  # [N,4,1,512], ch p = x[4j+p]
Q16 = np.clip(np.round(Zh / S_IN), -32768, 32767).astype(np.int16)                       # [N,2048] 원 샘플 순서
Q_chlast  = Q16.reshape(N, 512, 4)                       # 채널 마지막 = 센서 샘플 순서 그대로
Q_chfirst = np.ascontiguousarray(Q_chlast.transpose(0, 2, 1))   # 채널 우선 = buf[p*512+j] = x[4j+p]
n_sat = int((np.abs(np.round(Zh / S_IN)) > 32767).sum())
print(f"[INT16] scale 2^-12 | 포화 샘플 {n_sat}개 (+8.0 → 32767 = 7.99976, ORT QuantizeLinear와 동일 동작)")

# ── 3. PC 기준 출력 ──
E_poly = _run(SC, IC, OC, X_poly).astype(np.float32)
E_fp32 = _run(S32, I32, O32, np.ascontiguousarray(Zh[:, None, :])).astype(np.float32)
# 점검: int16로 바꾼 뒤 되돌린 입력으로도 같은 출력이 나오는가(= 모델 내부 입력 양자화와 우리 int16 변환 일치)
X_deq = np.ascontiguousarray((Q_chfirst.astype(np.float32) * S_IN)[:, :, None, :])
d_q = float(np.abs(_run(SC, IC, OC, X_deq) - E_poly).max())
cos = (E_poly * E_fp32).sum(1) / (np.linalg.norm(E_poly, axis=1) * np.linalg.norm(E_fp32, axis=1) + 1e-12)
print(f"[REF] poly vs FP32 cos median {np.median(cos):.5f} / min {cos.min():.5f}")
print(f"[CHECK] int16 왕복 입력 vs float 입력 출력 최대차 {d_q:.2e} → {'일치 ✅' if d_q == 0 else '⚠ 불일치 — 그대로 보내주세요'}")

# ── 4. 저장 ──
HIL = INT8_DIR / "hil_v1"; HIL.mkdir(exist_ok=True)
files = {
    "hil_inputs_float_poly.npy":   X_poly,     # stedgeai validate 입력용 (ONNX 모델 형식)
    "hil_outputs_ref_poly.npy":    E_poly,     # PC ORT 기준 출력 (보드와 비교)
    "hil_outputs_ref_fp32.npy":    E_fp32,     # 참고: FP32 출력
    "hil_inputs_int16_chlast.npy": Q_chlast,   # 펌웨어 시험용 (#15)
    "hil_inputs_int16_chfirst.npy": Q_chfirst, # 펌웨어 시험용 (#15)
}
for fn, arr in files.items(): np.save(HIL / fn, arr)
IDX.to_csv(HIL / "hil_index.csv", index=False, encoding="utf-8-sig")
sha = {fn: _sha256(HIL / fn) for fn in list(files) + ["hil_index.csv"]}
(HIL / "HIL_SUMMARY.json").write_text(json.dumps(dict(
    status="M2ENC_HIL_DATA_READY", created=datetime.datetime.now().isoformat(),
    model=P_CAND.name, model_sha256=CAND_SHA, decision="D-036", source="FULL_TEST only", learning_read=False,
    n_windows=N, per_record=PER_SEG * 3, input_scale="2^-12", clip=Q_IN, saturated_samples=n_sat,
    input_model_layout="[N,4,1,512] ch p = x[4j+p]", int16_roundtrip_maxdiff=d_q,
    ref_cos_vs_fp32=dict(median=float(np.median(cos)), min=float(cos.min())),
    files_sha256=sha, resolves=["#15 (pending firmware)", "#16 (pending board validate)"]),
    ensure_ascii=False, indent=2), encoding="utf-8")

ZIP = Path("/content/m2enc_hil_v1.zip")
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for fn in list(files) + ["hil_index.csv", "HIL_SUMMARY.json"]: z.write(HIL / fn, fn)
shutil.copy(P_CAND, "/content/m2enc_w8a16_poly_v1.onnx")
print(f"[SAVE] {HIL}\n[ZIP] {ZIP} ({ZIP.stat().st_size/1e6:.2f} MB)")
for fn, h in sha.items(): print(f"   {fn:32s} {h[:16]}…")
try:
    from google.colab import files as _f; _f.download(str(ZIP))
except Exception as e:
    print(f"⚠ 자동 다운로드 실패({e}) — 왼쪽 파일 탭에서 /content/m2enc_hil_v1.zip 직접 다운로드")
print("\nSTATUS: M2ENC_HIL_DATA_READY | LEARNING 사용: 없음 | 시도 소모 없음")


[MODEL] enc_w8a16_poly_pow2.onnx sha=df6e27f6945c2f66… (D-036)
[DATA] FULL_TEST 11개 레코드, 330윈도 | |max|=8.0000 | LEARNING 미사용
[INT16] scale 2^-12 | 포화 샘플 18761개 (+8.0 → 32767 = 7.99976, ORT QuantizeLinear와 동일 동작)
[REF] poly vs FP32 cos median 0.99994 / min 0.99959
[CHECK] int16 왕복 입력 vs float 입력 출력 최대차 0.00e+00 → 일치 ✅
[SAVE] /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/bridge_outputs/m2_enc_v1_train_42f861b9506e/int8_v1/hil_v1
[ZIP] /content/m2enc_hil_v1.zip (4.98 MB)
   hil_inputs_float_poly.npy        be21093b2e00ea62…
   hil_outputs_ref_poly.npy         1bc2d2b4ad6111ad…
   hil_outputs_ref_fp32.npy         6fc184ebe927f6cc…
   hil_inputs_int16_chlast.npy      ea04972175773e0e…
   hil_inputs_int16_chfirst.npy     36054567975c4802…
   hil_index.csv                    0677407ba45a4897…


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


STATUS: M2ENC_HIL_DATA_READY | LEARNING 사용: 없음 | 시도 소모 없음
